# EXP85 Phase 2 Complete Submission ? Standalone Notebook

This **single notebook contains the complete Python package source**, configs, and frozen Phase 2 comparison evidence copied into cells. It does not depend on the repository or adjacent `.py` files at run time: it materializes its embedded source into a fresh runtime folder so the original validated modules can execute unchanged.

You still need the exact three raw CSV inputs and a compatible Python environment with the pinned packages. The current pipeline also requires an **NVIDIA GeForce RTX 4050 on GPU 0** (as in the validated run); it deliberately fails rather than silently switching CatBoost to CPU. Input hashes are pinned to the validated Phase 2 data. The official competition score is not evaluated here.

The notebook is large because it embeds both the complete implementation and the frozen validation arrays. All code cells are initially unexecuted and have no saved outputs. Run cells from top to bottom; the final training cell performs all five folds and writes a uniquely named submission.

## Standalone runtime setup

Choose the raw-data folder using `PHASE2_DATA_DIR` if automatic discovery is not suitable. The notebook creates a unique writable runtime directory under the current working directory, restores its embedded code/config/reference bundle there, and imports only that restored copy.

In [ ]:
from datetime import datetime, timezone
import base64
import hashlib
import importlib.util
import io
import json
import os
import sys
import zipfile
from pathlib import Path

NOTEBOOK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUNTIME_ROOT = Path.cwd() / f"phase2_notebook_runtime_{NOTEBOOK_RUN_ID}"
PACKAGE_DIR = RUNTIME_ROOT / "champion" / "EXP85_PHASE2_COMPLETE_SUBMISSION"
FROZEN_DIR = RUNTIME_ROOT / "champion" / "EXP85_PHASE2_FROZEN_RELEASE"
for directory in (PACKAGE_DIR / "source", PACKAGE_DIR / "scripts", PACKAGE_DIR / "tests", PACKAGE_DIR / "config"):
    directory.mkdir(parents=True, exist_ok=True)


def _write_embedded_source(relative_path, source_text, expected_sha256):
    destination = (PACKAGE_DIR / relative_path).resolve()
    if not destination.is_relative_to(PACKAGE_DIR.resolve()):
        raise ValueError(f"Embedded path escapes runtime package: {relative_path}")
    source_bytes = source_text.encode("utf-8")
    actual = hashlib.sha256(source_bytes).hexdigest()
    if actual != expected_sha256:
        raise ValueError(f"Embedded source hash mismatch for {relative_path}: {actual}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(source_bytes)
    return actual


def _restore_embedded_evidence(archive_b64, expected_sha256):
    archive_bytes = base64.b64decode("".join(archive_b64.split()), validate=True)
    actual = hashlib.sha256(archive_bytes).hexdigest()
    if actual != expected_sha256:
        raise ValueError(f"Embedded frozen-evidence archive hash mismatch: {actual}")
    root = RUNTIME_ROOT.resolve()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        for member in archive.infolist():
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe embedded evidence path: {member.filename}")
        archive.extractall(root)
    return actual

print(f"Writable runtime: {RUNTIME_ROOT}")
print("Runtime starts empty; embedded project files are added in the following cells.")

## Complete embedded Python source

Each following source cell contains the **verbatim contents** of one original `.py` file inside a readable Python string. Executing a source cell verifies its SHA-256 and writes that exact source into the notebook-created runtime package. These cells only restore source; they do not train models.

### Embedded `source/data_validation.py`

In [ ]:
# Verbatim embedded source: source/data_validation.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import hashlib
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"
INPUT_NAMES = ("train.csv", "test.csv", "sample_submission.csv")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_frames(
    train: pd.DataFrame,
    test: pd.DataFrame,
    sample: pd.DataFrame,
    expected_target_hash: str | None = None,
) -> tuple[np.ndarray, dict[str, Any]]:
    if ID_COLUMN not in train or TARGET_COLUMN not in train:
        raise ValueError(f"Training input must include {ID_COLUMN} and {TARGET_COLUMN}.")
    if ID_COLUMN not in test or TARGET_COLUMN in test:
        raise ValueError(f"Test input must include {ID_COLUMN} and exclude {TARGET_COLUMN}.")
    if list(sample.columns) != [ID_COLUMN, TARGET_COLUMN]:
        raise ValueError(f"Unexpected sample schema: {sample.columns.tolist()}")
    expected_test_columns = [c for c in train.columns if c != TARGET_COLUMN]
    if expected_test_columns != list(test.columns):
        raise ValueError("Train predictors and test columns/order are incompatible.")
    if train.empty or test.empty or len(sample) != len(test):
        raise ValueError("Train, test, and sample inputs must be non-empty and aligned.")

    for label, frame in (("train", train), ("test", test), ("sample", sample)):
        if frame[ID_COLUMN].isna().any() or frame[ID_COLUMN].duplicated().any():
            raise ValueError(f"{label} IDs are missing or duplicated.")
    if not np.array_equal(test[ID_COLUMN].to_numpy(), sample[ID_COLUMN].to_numpy()):
        raise ValueError("Test and sample IDs/order differ.")

    target = pd.to_numeric(train[TARGET_COLUMN], errors="coerce")
    if target.isna().any() or not set(target.unique()).issubset({0, 1}):
        raise ValueError("Training TARGET must contain only non-missing binary 0/1 labels.")
    labels = target.to_numpy(dtype=np.int8)
    if np.unique(labels).size != 2:
        raise ValueError("Training TARGET must contain both classes.")
    if expected_target_hash is not None:
        actual = hashlib.sha256(labels.tobytes()).hexdigest()
        if actual != expected_target_hash:
            raise ValueError(
                f"Target-order fingerprint mismatch: expected {expected_target_hash}, got {actual}."
            )
    metadata = {
        "train_rows": int(len(train)),
        "test_rows": int(len(test)),
        "sample_rows": int(len(sample)),
        "train_columns": list(train.columns),
        "test_columns": list(test.columns),
        "sample_columns": list(sample.columns),
        "feature_columns": expected_test_columns,
        "class_counts": {
            str(int(value)): int(count)
            for value, count in zip(*np.unique(labels, return_counts=True))
        },
        "ids_unique": True,
        "test_ids_equal_sample_order": True,
        "train_test_predictor_schema_equal": True,
    }
    return labels, metadata


def load_inputs(
    data_dir: Path, expected_hashes: dict[str, str] | None = None
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, np.ndarray, dict[str, Any]]:
    paths = {name: data_dir / name for name in INPUT_NAMES}
    for name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(f"Required input is missing: {path}")
        if expected_hashes and name in expected_hashes:
            actual = sha256_file(path)
            if actual != expected_hashes[name]:
                raise ValueError(
                    f"{name} SHA-256 mismatch: expected {expected_hashes[name]}, got {actual}."
                )
    train, test, sample = (
        pd.read_csv(paths["train.csv"]),
        pd.read_csv(paths["test.csv"]),
        pd.read_csv(paths["sample_submission.csv"]),
    )
    labels, metadata = validate_frames(train, test, sample)
    metadata["input_hashes"] = {
        name: {"sha256": sha256_file(path), "size_bytes": path.stat().st_size}
        for name, path in paths.items()
    }
    return train, test, sample, labels, metadata
'''
_write_embedded_source('source/data_validation.py', _EMBEDDED_SOURCE_TEXT, '99c50ab3ba8a8d646c73cb04c77c88fb38915b41f475c0e4450e4c544204899c')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/preprocessing.py`

In [ ]:
# Verbatim embedded source: source/preprocessing.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.special import ndtr


CATEGORICAL_COLUMNS = (
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
)
ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"


def safe_ratio(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    result = np.full(numerator.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(numerator) & np.isfinite(denominator) & (denominator != 0)
    result[valid] = numerator[valid] / denominator[valid]
    return result


def build_rowwise_features(raw: pd.DataFrame) -> pd.DataFrame:
    missing_columns = set(CATEGORICAL_COLUMNS) - set(raw.columns)
    if missing_columns:
        raise ValueError(f"Missing categorical columns: {sorted(missing_columns)}")
    if ID_COLUMN not in raw or TARGET_COLUMN in raw:
        if TARGET_COLUMN in raw:
            raise ValueError("Feature builder must not receive the target column.")
        raise ValueError(f"Missing required ID column: {ID_COLUMN}")

    numeric = raw.drop(columns=[ID_COLUMN]).select_dtypes(include=[np.number]).copy()
    if np.isinf(numeric.to_numpy(dtype=np.float64, copy=False)).any():
        raise ValueError("Raw numeric features contain infinite values.")
    categorical = raw.loc[:, CATEGORICAL_COLUMNS].reset_index(drop=True).copy()
    categorical = categorical.fillna("MISSING").astype(str)
    features = numeric.reset_index(drop=True)

    for column in numeric.columns:
        features[f"NA_{column}"] = numeric[column].isna().to_numpy(dtype=np.float32)
    features["RAW_MISSING_COUNT"] = (
        raw.drop(columns=[ID_COLUMN]).isna().sum(axis=1).to_numpy(dtype=np.float32)
    )

    get = lambda name: raw[name].to_numpy(dtype=np.float64)
    income = get("AMT_INCOME_TOTAL")
    credit = get("AMT_CREDIT")
    annuity = get("AMT_ANNUITY")
    goods = get("AMT_GOODS_PRICE")
    family = raw["CNT_FAM_MEMBERS"].fillna(1.0).to_numpy(dtype=np.float64)
    birth = get("DAYS_BIRTH")
    emp_raw = get("DAYS_EMPLOYED")
    emp_clean = np.where(emp_raw == 365243.0, np.nan, emp_raw)
    registration = get("DAYS_REGISTRATION")
    id_published = get("DAYS_ID_PUBLISH")
    phone = get("DAYS_LAST_PHONE_CHANGE")
    goods_filled = np.where(np.isnan(goods), credit, goods)

    derived: dict[str, np.ndarray] = {
        "DAYS_EMPLOYED_CLEAN": emp_clean,
        "DAYS_EMPLOYED_SENTINEL": (emp_raw == 365243.0).astype(np.float32),
        "AGE_YEARS": -birth / 365.25,
        "EMPLOYED_YEARS": -emp_clean / 365.25,
        "EMPLOYED_TO_AGE": safe_ratio(emp_clean, birth + 1.0),
        "REGISTRATION_TO_AGE": safe_ratio(registration, birth + 1.0),
        "ID_PUBLISH_TO_AGE": safe_ratio(id_published, birth + 1.0),
        "PHONE_CHANGE_YEARS": -phone / 365.25,
        "CREDIT_TO_INCOME": safe_ratio(credit, income),
        "ANNUITY_TO_INCOME": safe_ratio(annuity, income),
        "CREDIT_TO_GOODS": safe_ratio(credit, goods_filled),
        "ANNUITY_TO_CREDIT": safe_ratio(annuity, credit),
        "CREDIT_TERM": safe_ratio(credit, annuity),
        "INCOME_PER_FAMILY": safe_ratio(income, family),
        "CREDIT_PER_FAMILY": safe_ratio(credit, family),
        "ANNUITY_PER_FAMILY": safe_ratio(annuity, family),
        "DISPOSABLE_INCOME": income - 12.0 * annuity,
        "DISPOSABLE_INCOME_TO_INCOME": safe_ratio(income - 12.0 * annuity, income),
        "DEBT_SERVICE_COVERAGE": safe_ratio(income / 12.0, annuity),
        "CREDIT_TERM_ROUND_1": safe_ratio(credit, annuity + 1.0).round(1),
        "CREDIT_TO_GOODS_ROUND_2": safe_ratio(credit, goods_filled + 1.0).round(2),
    }

    ext = np.column_stack([get("EXT_SOURCE_1"), get("EXT_SOURCE_2"), get("EXT_SOURCE_3")])
    ext_count = np.isfinite(ext).sum(axis=1)
    ext_sum = np.where(np.isfinite(ext), ext, 0.0).sum(axis=1)
    ext_mean = np.divide(
        ext_sum,
        ext_count,
        out=np.full(len(ext), np.nan, dtype=np.float64),
        where=ext_count > 0,
    )
    ext_min = np.where(np.isfinite(ext), ext, np.inf).min(axis=1)
    ext_max = np.where(np.isfinite(ext), ext, -np.inf).max(axis=1)
    ext_min[ext_count == 0] = np.nan
    ext_max[ext_count == 0] = np.nan
    ext_variance = np.divide(
        np.where(np.isfinite(ext), (ext - ext_mean[:, None]) ** 2, 0.0).sum(axis=1),
        ext_count,
        out=np.full(len(ext), np.nan, dtype=np.float64),
        where=ext_count > 0,
    )
    ext_std = np.sqrt(ext_variance)
    ext_product = np.where(np.isfinite(ext), ext, 1.0).prod(axis=1)
    ext_product[ext_count == 0] = np.nan
    derived.update(
        {
            "EXT_MEAN": ext_mean,
            "EXT_MIN": ext_min,
            "EXT_MAX": ext_max,
            "EXT_STD": np.where(ext_count == 0, 0.0, ext_std),
            "EXT_RANGE": ext_max - ext_min,
            "EXT_COUNT": ext_count.astype(np.float32),
            "EXT_PRODUCT": ext_product,
        }
    )
    for left, right in ((0, 1), (0, 2), (1, 2)):
        derived[f"EXT_DIFF_{left}{right}"] = ext[:, left] - ext[:, right]
        derived[f"EXT_PRODUCT_{left}{right}"] = ext[:, left] * ext[:, right]
        derived[f"EXT_RATIO_{left}{right}"] = safe_ratio(ext[:, left], ext[:, right])

    term_years = np.clip(safe_ratio(credit, annuity + 1.0) / 12.0, 0.1, 10.0)
    disposable_ratio = np.maximum(0.01, safe_ratio(income / 12.0, annuity + 1.0))
    for mu, sigma, suffix in ((0.02, 0.20, "20"), (0.03, 0.25, "25")):
        distance = (
            np.log(disposable_ratio) + (mu - 0.5 * sigma**2) * term_years
        ) / (sigma * np.sqrt(term_years))
        derived[f"MERTON_DISTANCE_{suffix}"] = np.clip(distance, -10.0, 10.0)
        derived[f"MERTON_PD_{suffix}"] = ndtr(-distance)

    bureau_columns = [
        column for column in raw.columns if column.startswith("AMT_REQ_CREDIT_BUREAU_")
    ]
    bureau = raw[bureau_columns]
    derived["BUREAU_INQUIRY_SUM"] = bureau.sum(axis=1, min_count=1).to_numpy()
    derived["INCOME_LOG1P"] = np.log1p(np.maximum(income, 0.0))
    derived["CREDIT_LOG1P"] = np.log1p(np.maximum(credit, 0.0))
    derived["ANNUITY_LOG1P"] = np.log1p(np.maximum(annuity, 0.0))

    derived_frame = pd.DataFrame(derived)
    if np.isinf(derived_frame.to_numpy(dtype=np.float64, copy=False)).any():
        raise ValueError("Feature engineering produced infinite values.")
    features = pd.concat([features, derived_frame], axis=1)
    features = features.loc[:, ~features.columns.duplicated()]
    for column in features.columns:
        if not pd.api.types.is_numeric_dtype(features[column]):
            raise TypeError(f"Non-numeric derived feature: {column}")
    return pd.concat([features, categorical], axis=1).reset_index(drop=True)


def make_lightgbm_frames(
    train_features: pd.DataFrame, test_features: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    numeric_columns = [
        column for column in train_features.columns if column not in CATEGORICAL_COLUMNS
    ]
    train_parts = [train_features[numeric_columns].astype(np.float32)]
    test_parts = [test_features[numeric_columns].astype(np.float32)]
    for column in CATEGORICAL_COLUMNS:
        levels = sorted(train_features[column].astype(str).unique().tolist())
        train_cat = pd.Categorical(train_features[column].astype(str), categories=levels)
        test_cat = pd.Categorical(test_features[column].astype(str), categories=levels)
        category_columns = [f"CAT_{column}_{index}" for index in range(len(levels))]
        train_dummy = pd.get_dummies(train_cat, dtype=np.float32)
        test_dummy = pd.get_dummies(test_cat, dtype=np.float32)
        train_dummy.columns = category_columns
        test_dummy.columns = category_columns
        train_parts.append(train_dummy)
        test_parts.append(test_dummy)
    X_train = pd.concat(train_parts, axis=1)
    X_test = pd.concat(test_parts, axis=1).reindex(columns=X_train.columns, fill_value=0.0)
    if not X_train.columns.is_unique or list(X_train.columns) != list(X_test.columns):
        raise ValueError("LightGBM train/test feature schemas differ.")
    if any(not column.replace("_", "").isalnum() for column in X_train.columns):
        raise ValueError("LightGBM feature names must be alphanumeric with underscores.")
    if np.isinf(X_train.to_numpy(dtype=np.float64, copy=False)).any() or np.isinf(
        X_test.to_numpy(dtype=np.float64, copy=False)
    ).any():
        raise ValueError("LightGBM features contain infinite values.")
    return X_train, X_test
'''
_write_embedded_source('source/preprocessing.py', _EMBEDDED_SOURCE_TEXT, '1c24f11f3e8f9d26d8e4a5821f7052a78a58deb337991e0847b93560fec79c7e')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/phase1_features.py`

In [ ]:
# Verbatim embedded source: source/phase1_features.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd

from preprocessing import TARGET_COLUMN


FEE_AMOUNT = 129.5
PRODUCT_RATE = 0.0132


def build_decoded_loan_features(
    train_raw: pd.DataFrame,
    test_raw: pd.DataFrame,
    pooled_frequency_reference: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    if not pooled_frequency_reference:
        raise ValueError(
            "Selected seed-42 Phase 2 recipe uses pooled unlabeled train/test frequencies."
        )
    combined = pd.concat(
        [train_raw.drop(columns=[TARGET_COLUMN], errors="ignore"), test_raw],
        ignore_index=True,
    )
    credit = combined["AMT_CREDIT"].to_numpy(dtype=np.float64)
    annuity = combined["AMT_ANNUITY"].to_numpy(dtype=np.float64)
    goods = combined["AMT_GOODS_PRICE"].to_numpy(dtype=np.float64)
    contract = combined["NAME_CONTRACT_TYPE"].astype("string").to_numpy()

    raw_ratio = np.full(len(combined), np.nan, dtype=np.float64)
    fee_ratio = np.full(len(combined), np.nan, dtype=np.float64)
    valid_credit = np.isfinite(credit) & (credit != 0.0)
    raw_ratio[valid_credit] = annuity[valid_credit] / credit[valid_credit]
    fee_ratio[valid_credit] = (annuity[valid_credit] - FEE_AMOUNT) / credit[valid_credit]

    raw_key = pd.Series(raw_ratio).round(5)
    fee_key = pd.Series(fee_ratio).round(5)
    raw_counts = raw_key.map(raw_key.value_counts()).fillna(0).to_numpy()
    fee_counts = fee_key.map(fee_key.value_counts()).fillna(0).to_numpy()
    fee_decoded = (
        (contract == "Cash loans")
        & (fee_counts > raw_counts)
        & np.isfinite(raw_ratio)
    )
    decoded_ratio = np.where(fee_decoded, fee_ratio, raw_ratio)

    goods_valid = np.isfinite(goods) & (goods != 0.0)
    credit_goods_ratio = np.full(len(combined), np.nan, dtype=np.float64)
    credit_goods_ratio[goods_valid] = credit[goods_valid] / goods[goods_valid]
    implied_product = (credit_goods_ratio - 1.0) / PRODUCT_RATE
    nearest_product = np.rint(implied_product)
    product_valid = (
        (contract == "Cash loans")
        & np.isfinite(implied_product)
        & (credit > goods)
        & (np.abs(implied_product - nearest_product) < 0.1)
    )

    features = pd.DataFrame(
        {
            "FEE_DECODED": fee_decoded.astype(np.float32),
            "FEE_RAW_RATIO_COUNT": np.log1p(raw_counts).astype(np.float32),
            "FEE_ADJUSTED_RATIO_COUNT": np.log1p(fee_counts).astype(np.float32),
            "ANNUITY_CREDIT_RATIO_FEE_ADJUSTED": fee_ratio,
            "ANNUITY_CREDIT_RATIO_DECODED": decoded_ratio,
            "ANNUITY_CREDIT_RATIO_DECODED_ROUND_3": np.round(decoded_ratio, 3),
            "ANNUITY_CREDIT_RATIO_DECODED_ROUND_4": np.round(decoded_ratio, 4),
            "ANNUITY_CREDIT_RATIO_DECODED_ROUND_5": np.round(decoded_ratio, 5),
            "CREDIT_GOODS_RATIO": credit_goods_ratio,
            "PRODUCT_KEY_VALID": product_valid.astype(np.float32),
            "PRODUCT_KEY_K": np.where(product_valid, nearest_product, np.nan),
            "PRODUCT_KEY_DISTANCE": np.where(
                product_valid, implied_product - nearest_product, np.nan
            ),
        }
    )
    if np.isinf(features.to_numpy(dtype=np.float64, copy=False)).any():
        raise ValueError("Decoded loan features contain infinite values.")

    n_train = len(train_raw)
    train_features = features.iloc[:n_train].reset_index(drop=True)
    test_features = features.iloc[n_train:].reset_index(drop=True)
    diagnostics = {
        "fee_amount": FEE_AMOUNT,
        "product_rate": PRODUCT_RATE,
        "fee_decoded_rows": int(fee_decoded.sum()),
        "fee_decoded_cash_loan_fraction": float(fee_decoded[contract == "Cash loans"].mean()),
        "product_key_valid_rows": int(product_valid.sum()),
        "product_key_valid_fraction_of_cash_loans": float(
            product_valid[contract == "Cash loans"].mean()
        ),
        "feature_names": features.columns.tolist(),
        "frequency_counts_use_train_and_test_covariates": True,
        "frequency_counts_use_target": False,
    }
    return train_features, test_features, diagnostics
'''
_write_embedded_source('source/phase1_features.py', _EMBEDDED_SOURCE_TEXT, '36957fd898c92705f11b22d3bf70cc92d73656813fbf683b0136d5a1452baa6d')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/phase2_target_encoding.py`

In [ ]:
# Verbatim embedded source: source/phase2_target_encoding.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold


MISSING_KEY = "__MISSING_KEY__"
KEY_SPECS = (
    ("FEE_Q3", "ANNUITY_CREDIT_RATIO_DECODED_ROUND_3", 100.0),
    ("FEE_Q4", "ANNUITY_CREDIT_RATIO_DECODED_ROUND_4", 30.0),
    ("FEE_Q5", "ANNUITY_CREDIT_RATIO_DECODED_ROUND_5", 30.0),
    ("PRODUCT_K", "PRODUCT_KEY_K", 50.0),
)
INNER_SPLITS = 5
INNER_SEED = 7


def key_arrays(
    fee_train: pd.DataFrame, fee_test: pd.DataFrame
) -> dict[str, np.ndarray]:
    combined = pd.concat([fee_train, fee_test], ignore_index=True)
    result = {}
    for name, column, _ in KEY_SPECS:
        values = combined[column].round(0 if name == "PRODUCT_K" else int(column[-1]))
        result[name] = values.astype("string").fillna(MISSING_KEY).to_numpy(dtype=str)
    return result


def encode_keys(
    keys: np.ndarray,
    labels: np.ndarray,
    fit_indices: np.ndarray,
    apply_indices: np.ndarray,
    key_name: str,
    smoothing: float,
) -> tuple[np.ndarray, np.ndarray]:
    keys = np.asarray(keys)
    labels = np.asarray(labels)
    fit_indices = np.asarray(fit_indices, dtype=np.int64)
    apply_indices = np.asarray(apply_indices, dtype=np.int64)
    if keys.ndim != 1 or labels.ndim != 1:
        raise ValueError("Target-encoding keys and labels must be one-dimensional.")
    if len(keys) < len(labels) or not np.isfinite(labels).all():
        raise ValueError("Invalid target-encoding key or label arrays.")
    if fit_indices.size == 0 or np.any(fit_indices < 0) or np.any(fit_indices >= len(labels)):
        raise ValueError("Target-encoding fit indices are empty or out of label bounds.")
    if np.any(apply_indices < 0) or np.any(apply_indices >= len(keys)):
        raise ValueError("Target-encoding apply indices are out of key bounds.")
    if np.intersect1d(fit_indices, apply_indices).size:
        raise ValueError("Target-encoding fit and apply indices overlap.")
    if not np.isfinite(smoothing) or smoothing <= 0:
        raise ValueError("Target-encoding smoothing must be positive and finite.")

    prior = float(np.mean(labels[fit_indices]))
    fit = pd.DataFrame(
        {"key": keys[fit_indices], "target": labels[fit_indices].astype(np.float64)}
    )
    grouped = fit.groupby("key", sort=False)["target"].agg(["sum", "count"])
    smoothed = (grouped["sum"] + smoothing * prior) / (grouped["count"] + smoothing)
    apply_keys = pd.Series(keys[apply_indices])
    encoded = apply_keys.map(smoothed).fillna(prior).to_numpy(dtype=np.float32)
    counts = apply_keys.map(grouped["count"]).fillna(0).to_numpy(dtype=np.float32)
    return encoded, counts


def nested_training_encodings(
    keys_by_name: dict[str, np.ndarray],
    labels: np.ndarray,
    outer_train_indices: np.ndarray,
) -> pd.DataFrame:
    labels = np.asarray(labels)
    outer_train_indices = np.asarray(outer_train_indices, dtype=np.int64)
    if np.any(outer_train_indices < 0) or np.any(outer_train_indices >= len(labels)):
        raise ValueError("Outer training indices are out of label bounds.")
    output = np.full((len(outer_train_indices), 2 * len(KEY_SPECS)), np.nan, dtype=np.float32)
    inner = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=INNER_SEED)
    local_y = labels[outer_train_indices]
    for inner_fit_local, inner_apply_local in inner.split(
        np.zeros(len(outer_train_indices)), local_y
    ):
        inner_fit = outer_train_indices[inner_fit_local]
        inner_apply = outer_train_indices[inner_apply_local]
        if np.intersect1d(inner_fit, inner_apply).size:
            raise RuntimeError("Nested target-encoding split overlaps.")
        for key_index, (name, _, smoothing) in enumerate(KEY_SPECS):
            encoded, count = encode_keys(
                keys_by_name[name], labels, inner_fit, inner_apply, name, smoothing
            )
            output[inner_apply_local, 2 * key_index] = encoded
            output[inner_apply_local, 2 * key_index + 1] = np.log1p(count)
    if not np.isfinite(output).all():
        raise ValueError("Nested target encodings are incomplete or non-finite.")
    columns = [
        column
        for name, _, _ in KEY_SPECS
        for column in (f"TE_{name}", f"LOG_COUNT_{name}")
    ]
    return pd.DataFrame(output, columns=columns)


def apply_training_encodings(
    keys_by_name: dict[str, np.ndarray],
    labels: np.ndarray,
    fit_indices: np.ndarray,
    apply_indices: np.ndarray,
) -> pd.DataFrame:
    values = {}
    for name, _, smoothing in KEY_SPECS:
        encoded, count = encode_keys(
            keys_by_name[name], labels, fit_indices, apply_indices, name, smoothing
        )
        values[f"TE_{name}"] = encoded
        values[f"LOG_COUNT_{name}"] = np.log1p(count)
    return pd.DataFrame(values).reset_index(drop=True)
'''
_write_embedded_source('source/phase2_target_encoding.py', _EMBEDDED_SOURCE_TEXT, '2829fc049fe071f62691c9a4d72cd051b56abbd31f4c3538339340c3eb998d51')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/folds.py`

In [ ]:
# Verbatim embedded source: source/folds.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import numpy as np
from sklearn.model_selection import StratifiedKFold


def build_folds(
    labels: np.ndarray, n_splits: int = 5, seed: int = 42
) -> list[tuple[np.ndarray, np.ndarray]]:
    labels = np.asarray(labels)
    if labels.ndim != 1 or len(labels) == 0:
        raise ValueError("Fold labels must be a non-empty vector.")
    if n_splits < 2:
        raise ValueError("At least two folds are required.")
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [
        (fit.astype(np.int64), valid.astype(np.int64))
        for fit, valid in splitter.split(np.zeros(len(labels)), labels)
    ]


def fold_id_vector(
    labels: np.ndarray, folds: list[tuple[np.ndarray, np.ndarray]]
) -> np.ndarray:
    result = np.full(len(labels), -1, dtype=np.int16)
    for fold_number, (_, valid) in enumerate(folds, 1):
        if np.any(result[valid] != -1):
            raise ValueError("Validation fold assignments overlap.")
        result[valid] = fold_number
    if np.any(result < 1):
        raise ValueError("Validation folds do not cover every training row.")
    return result
'''
_write_embedded_source('source/folds.py', _EMBEDDED_SOURCE_TEXT, 'cb36245b9e999ba85bc022d2e86f20dd301498073f37ab3119929ccf6b5e9594')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/prediction_blender.py`

In [ ]:
# Verbatim embedded source: source/prediction_blender.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import numpy as np


def _validate_prediction(values: np.ndarray, name: str) -> np.ndarray:
    values = np.asarray(values)
    if values.ndim != 1:
        raise ValueError(f"{name} must be a one-dimensional prediction vector.")
    if not np.issubdtype(values.dtype, np.number) or not np.isfinite(values).all():
        raise ValueError(f"{name} must contain finite numeric predictions.")
    if np.any((values < 0.0) | (values > 1.0)):
        raise ValueError(f"{name} probabilities must lie in [0, 1].")
    return values


def combine_phase2_predictions(
    lightgbm_predictions: np.ndarray, catboost_predictions: np.ndarray
) -> np.ndarray:
    lightgbm_predictions = _validate_prediction(lightgbm_predictions, "LightGBM")
    catboost_predictions = _validate_prediction(catboost_predictions, "CatBoost")
    if lightgbm_predictions.shape != catboost_predictions.shape:
        raise ValueError("LightGBM and CatBoost prediction shapes differ.")
    return (
        lightgbm_predictions.astype(np.float64)
        + catboost_predictions.astype(np.float64)
    ) / 2.0


def mean_fold_predictions(predictions: list[np.ndarray]) -> np.ndarray:
    if not predictions:
        raise ValueError("No fold predictions were provided.")
    arrays = [_validate_prediction(value, f"fold_{index}") for index, value in enumerate(predictions, 1)]
    if any(array.shape != arrays[0].shape for array in arrays):
        raise ValueError("Fold prediction shapes differ.")
    return np.mean(np.stack(arrays), axis=0, dtype=np.float64).astype(np.float32)
'''
_write_embedded_source('source/prediction_blender.py', _EMBEDDED_SOURCE_TEXT, 'b583adfb33f3c414a38f426fce30f2da05bec82f52dd430c43af18bb57c77a7a')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/submission_builder.py`

In [ ]:
# Verbatim embedded source: source/submission_builder.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"


def validate_submission(
    candidate: pd.DataFrame,
    test: pd.DataFrame,
    sample: pd.DataFrame,
) -> dict[str, Any]:
    if list(candidate.columns) != list(sample.columns):
        raise ValueError("Submission columns/order differ from sample_submission.csv.")
    if list(candidate.columns) != [ID_COLUMN, TARGET_COLUMN]:
        raise ValueError("Submission schema must be SK_ID_CURR,TARGET.")
    if len(candidate) != len(test) or len(candidate) != len(sample):
        raise ValueError("Submission row count differs from test/sample input.")
    if candidate[ID_COLUMN].isna().any() or candidate[ID_COLUMN].duplicated().any():
        raise ValueError("Submission IDs are missing or duplicated.")
    if not np.array_equal(candidate[ID_COLUMN].to_numpy(), test[ID_COLUMN].to_numpy()):
        raise ValueError("Submission IDs/order differ from test.csv.")
    if not np.array_equal(candidate[ID_COLUMN].to_numpy(), sample[ID_COLUMN].to_numpy()):
        raise ValueError("Submission IDs/order differ from sample_submission.csv.")
    values = pd.to_numeric(candidate[TARGET_COLUMN], errors="coerce").to_numpy(
        dtype=np.float64
    )
    if not np.isfinite(values).all() or np.any((values < 0.0) | (values > 1.0)):
        raise ValueError("Submission contains non-finite or out-of-range probabilities.")
    return {
        "status": "PASS",
        "rows": int(len(candidate)),
        "schema": list(candidate.columns),
        "ids_unique": True,
        "ids_match_test_order": True,
        "ids_match_sample_order": True,
        "probabilities_finite": True,
        "probability_min": float(values.min()),
        "probability_max": float(values.max()),
    }


def build_submission(
    test: pd.DataFrame,
    sample: pd.DataFrame,
    predictions: np.ndarray,
    path: Path,
) -> dict[str, Any]:
    predictions = np.asarray(predictions, dtype=np.float64)
    if predictions.ndim != 1 or len(predictions) != len(test):
        raise ValueError("Final predictions must be a vector aligned to test rows.")
    candidate = pd.DataFrame(
        {ID_COLUMN: test[ID_COLUMN].to_numpy(), TARGET_COLUMN: predictions}
    )
    validate_submission(candidate, test, sample)
    path.parent.mkdir(parents=True, exist_ok=True)
    candidate.to_csv(path, index=False, float_format="%.10f")
    read_back = pd.read_csv(path)
    return validate_submission(read_back, test, sample)
'''
_write_embedded_source('source/submission_builder.py', _EMBEDDED_SOURCE_TEXT, 'a59fe66428078a5bbfc0911997a8ec5f6b78c1e6b50c498f25bd88e7902a345e')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/gpu_telemetry.py`

In [ ]:
# Verbatim embedded source: source/gpu_telemetry.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import subprocess
import threading
from typing import Any


def gpu_snapshot() -> dict[str, Any]:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu",
            "--format=csv,noheader",
        ],
        check=True,
        capture_output=True,
        text=True,
        timeout=20,
    )
    return {"available": True, "query": result.stdout.strip()}


def require_gpu_preflight() -> dict[str, Any]:
    snapshot = gpu_snapshot()
    if "NVIDIA GeForce RTX 4050" not in snapshot["query"]:
        raise RuntimeError(f"Expected RTX 4050 GPU; detected: {snapshot['query']}")
    from catboost import CatBoostClassifier
    import numpy as np
    import pandas as pd

    frame = pd.DataFrame(
        {
            "numeric": np.arange(100, dtype=np.float32),
            "category": pd.Series(["a", "b"] * 50, dtype="object"),
        }
    )
    labels = np.asarray([0, 1] * 50, dtype=np.int8)
    model = CatBoostClassifier(
        iterations=2,
        depth=2,
        task_type="GPU",
        devices="0",
        allow_writing_files=False,
        verbose=False,
        thread_count=1,
    )
    model.fit(frame, labels, cat_features=["category"], verbose=False)
    return {
        "gpu_snapshot": snapshot,
        "catboost_gpu_smoke_test": "PASS",
        "catboost_gpu_task_type": model.get_params()["task_type"],
        "catboost_gpu_devices": model.get_params()["devices"],
        "smoke_test_tree_count": int(model.tree_count_),
    }


class GpuMonitor:
    def __init__(self, interval_seconds: float = 1.0) -> None:
        self.interval_seconds = interval_seconds
        self.samples: list[dict[str, Any]] = []
        self.errors: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def _sample(self) -> None:
        try:
            result = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=timestamp,name,utilization.gpu,memory.used",
                    "--format=csv,noheader",
                ],
                check=True,
                capture_output=True,
                text=True,
                timeout=10,
            )
            self.samples.append({"query": result.stdout.strip()})
        except (OSError, subprocess.SubprocessError) as error:
            self.errors.append(f"{type(error).__name__}: {error}")

    def _run(self) -> None:
        while not self._stop.wait(self.interval_seconds):
            self._sample()

    def __enter__(self) -> GpuMonitor:
        self._sample()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type: object, exc: object, traceback: object) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=15)
            if self._thread.is_alive():
                self.errors.append("GPU telemetry sampling thread did not stop.")
        self._sample()

    def report(self) -> dict[str, Any]:
        return {
            "samples": self.samples,
            "sample_count": len(self.samples),
            "errors": self.errors,
        }
'''
_write_embedded_source('source/gpu_telemetry.py', _EMBEDDED_SOURCE_TEXT, '7f4d1cdac15a095297c01a9e34446b4aea5cdcf4ac33dca13e1f8ce533dbedf7')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/train_lightgbm.py`

In [ ]:
# Verbatim embedded source: source/train_lightgbm.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

from pathlib import Path
from typing import Any

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def train_lightgbm_fold(
    X_train: pd.DataFrame,
    labels: np.ndarray,
    X_test: pd.DataFrame,
    fit_indices: np.ndarray,
    valid_indices: np.ndarray,
    fold: int,
    settings: dict[str, Any],
    seed: int,
    model_path: Path,
) -> dict[str, Any]:
    params = {
        key: value
        for key, value in settings.items()
        if key not in ("device", "num_boost_round", "early_stopping_rounds")
    }
    params["seed"] = seed
    params["num_threads"] = min(int(settings["num_threads"]), 8)
    train_set = lgb.Dataset(
        X_train.iloc[fit_indices], label=labels[fit_indices], free_raw_data=True
    )
    valid_set = lgb.Dataset(
        X_train.iloc[valid_indices],
        label=labels[valid_indices],
        reference=train_set,
        free_raw_data=True,
    )
    model = lgb.train(
        params,
        train_set,
        num_boost_round=int(settings["num_boost_round"]),
        valid_sets=[valid_set],
        callbacks=[
            lgb.early_stopping(
                int(settings["early_stopping_rounds"]), verbose=False
            )
        ],
    )
    valid_prediction = model.predict(
        X_train.iloc[valid_indices], num_iteration=model.best_iteration
    )
    test_prediction = model.predict(X_test, num_iteration=model.best_iteration)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(model_path), num_iteration=model.best_iteration)
    importance = pd.DataFrame(
        {
            "feature": X_train.columns,
            "gain": model.feature_importance(importance_type="gain"),
            "split": model.feature_importance(importance_type="split"),
            "fold": fold,
        }
    )
    return {
        "valid_prediction": valid_prediction.astype(np.float32),
        "test_prediction": test_prediction.astype(np.float32),
        "best_iteration": int(model.best_iteration),
        "auc": float(roc_auc_score(labels[valid_indices], valid_prediction)),
        "importance": importance,
        "device": "CPU",
        "model_path": str(model_path),
    }
'''
_write_embedded_source('source/train_lightgbm.py', _EMBEDDED_SOURCE_TEXT, '3c7f702c5f22ee57ce0f479ca806a247ae3c5c7e919428d69552cdd2071753d8')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/train_catboost.py`

In [ ]:
# Verbatim embedded source: source/train_catboost.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

from gpu_telemetry import GpuMonitor


def train_catboost_fold(
    X_train: pd.DataFrame,
    labels: np.ndarray,
    X_test: pd.DataFrame,
    fit_indices: np.ndarray,
    valid_indices: np.ndarray,
    fold: int,
    settings: dict[str, Any],
    seed: int,
    model_path: Path,
) -> dict[str, Any]:
    if settings.get("task_type") != "GPU" or settings.get("devices") != "0":
        raise ValueError("Selected Phase 2 CatBoost configuration requires GPU 0.")
    model = CatBoostClassifier(
        iterations=int(settings["iterations"]),
        depth=int(settings["depth"]),
        learning_rate=float(settings["learning_rate"]),
        loss_function=settings["loss_function"],
        eval_metric=settings["eval_metric"],
        l2_leaf_reg=float(settings["l2_leaf_reg"]),
        random_strength=float(settings["random_strength"]),
        bootstrap_type=settings["bootstrap_type"],
        subsample=float(settings["subsample"]),
        border_count=int(settings["border_count"]),
        task_type=settings["task_type"],
        devices=settings["devices"],
        random_seed=seed,
        allow_writing_files=False,
        verbose=False,
        thread_count=int(settings["num_threads"]),
    )
    monitor = GpuMonitor()
    with monitor:
        model.fit(
            X_train.iloc[fit_indices],
            labels[fit_indices],
            cat_features=list(settings["categorical_columns"]),
            eval_set=(X_train.iloc[valid_indices], labels[valid_indices]),
            early_stopping_rounds=int(settings["early_stopping_rounds"]),
            verbose=False,
        )
    valid_prediction = model.predict_proba(X_train.iloc[valid_indices])[:, 1]
    test_prediction = model.predict_proba(X_test)[:, 1]
    model_path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(model_path))
    importance = pd.DataFrame(
        {
            "feature": X_train.columns,
            "importance": model.get_feature_importance(),
            "fold": fold,
        }
    )
    telemetry = monitor.report()
    return {
        "valid_prediction": valid_prediction.astype(np.float32),
        "test_prediction": test_prediction.astype(np.float32),
        "best_iteration": int(model.best_iteration_),
        "auc": float(roc_auc_score(labels[valid_indices], valid_prediction)),
        "importance": importance,
        "device": "GPU:0",
        "model_path": str(model_path),
        "gpu_telemetry": telemetry,
    }
'''
_write_embedded_source('source/train_catboost.py', _EMBEDDED_SOURCE_TEXT, 'bdaf6b3e895d1857ef77cb8a3e4717b870fd63d50a454be823a467f96194c48e')
del _EMBEDDED_SOURCE_TEXT


### Embedded `source/pipeline.py`

In [ ]:
# Verbatim embedded source: source/pipeline.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from data_validation import load_inputs, sha256_file
from folds import build_folds, fold_id_vector
from gpu_telemetry import gpu_snapshot, require_gpu_preflight
from phase1_features import build_decoded_loan_features
from phase2_target_encoding import (
    INNER_SEED,
    INNER_SPLITS,
    KEY_SPECS,
    apply_training_encodings,
    key_arrays,
    nested_training_encodings,
)
from prediction_blender import combine_phase2_predictions, mean_fold_predictions
from preprocessing import CATEGORICAL_COLUMNS, ID_COLUMN, TARGET_COLUMN, build_rowwise_features, make_lightgbm_frames
from submission_builder import build_submission, validate_submission
from train_catboost import train_catboost_fold
from train_lightgbm import train_lightgbm_fold


PACKAGE_DIR = Path(__file__).resolve().parents[1]
REPO_ROOT = PACKAGE_DIR.parents[1]
DEFAULT_DATA_DIR = Path(r"D:\gci\inpuy")
FROZEN_DIR = PACKAGE_DIR.parent / "EXP85_PHASE2_FROZEN_RELEASE"
REFERENCE_DIR = FROZEN_DIR / "EVIDENCE" / "original_runs" / "phase_02"
REFERENCE_CANDIDATE = FROZEN_DIR / "CANDIDATE" / "submission_key_target_encoding.csv"
EXPECTED_CANDIDATE_SHA256 = (
    "2470cecc0819d696b03b8d0fcc80d7b91b680986e9c7fe877a57a1684a7411fa"
)


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def file_inventory(paths: list[Path], root: Path) -> dict[str, dict[str, Any]]:
    result = {}
    for path in sorted(paths):
        result[path.relative_to(root).as_posix()] = {
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        }
    return result


def package_versions() -> dict[str, str]:
    result = {"python": platform.python_version()}
    for name in ("numpy", "pandas", "scikit-learn", "lightgbm", "catboost", "scipy"):
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = "MISSING"
    return result


def git_evidence() -> dict[str, Any]:
    def command(args: list[str]) -> str | None:
        try:
            result = subprocess.run(
                ["git", "-C", str(REPO_ROOT), *args],
                check=True,
                capture_output=True,
                text=True,
                timeout=20,
            )
        except (OSError, subprocess.SubprocessError) as error:
            return f"ERROR: {type(error).__name__}: {error}"
        return result.stdout.strip()

    return {
        "repository": str(REPO_ROOT),
        "commit": command(["rev-parse", "HEAD"]),
        "branch": command(["branch", "--show-current"]),
        "status": command(["status", "--short"]),
    }


def verify_frozen_reference(data_dir: Path) -> dict[str, Any]:
    if not REFERENCE_DIR.is_dir() or not REFERENCE_CANDIDATE.is_file():
        raise FileNotFoundError(
            "Frozen Phase 2 reference artifacts are missing; reproduction gate cannot run."
        )
    release_manifest = read_json(FROZEN_DIR / "FREEZE_MANIFEST.json")
    candidate_hash = sha256_file(REFERENCE_CANDIDATE)
    if candidate_hash != EXPECTED_CANDIDATE_SHA256:
        raise ValueError("Frozen Phase 2 candidate hash changed.")

    reference_manifest_path = REFERENCE_DIR / "sha256_manifest.json"
    reference_manifest = read_json(reference_manifest_path)
    for relative, record in reference_manifest["outputs"].items():
        path = REFERENCE_DIR / relative
        if not path.is_file() or sha256_file(path) != record["sha256"]:
            raise ValueError(f"Frozen Phase 2 reference artifact hash mismatch: {path}")
    expected_input = release_manifest["raw_inputs"]
    for filename, path in (
        ("train.csv", data_dir / "train.csv"),
        ("test.csv", data_dir / "test.csv"),
        ("sample_submission.csv", data_dir / "sample_submission.csv"),
    ):
        expected = expected_input[filename]["sha256"]
        actual = sha256_file(path)
        if actual != expected:
            raise ValueError(
                f"Raw {filename} differs from the frozen Phase 2 input hash."
            )
    summary = read_json(REFERENCE_DIR / "experiment_summary.json")
    return {
        "frozen_candidate_sha256": candidate_hash,
        "reference_manifest_sha256": sha256_file(reference_manifest_path),
        "frozen_release_manifest_sha256": sha256_file(FROZEN_DIR / "FREEZE_MANIFEST.json"),
        "candidate_oof_auc": summary["auc_comparison"]["blend"][
            "full_oof_auc_fee_plus_nested_te"
        ],
        "candidate_oof_by_fold": [
            row["auc"]["fee_plus_nested_te"]["blend"]
            for row in summary["paired_cv_by_fold"]
        ],
        "reference_files_verified": len(reference_manifest["outputs"]),
        "raw_inputs_match_frozen_hashes": True,
    }


def load_reference_arrays() -> tuple[dict[str, np.ndarray], np.ndarray, list[tuple[np.ndarray, np.ndarray]]]:
    arrays = {
        name: np.load(REFERENCE_DIR / f"oof_{name}.npy", allow_pickle=False)
        for name in ("lightgbm", "catboost")
    }
    arrays["blend"] = np.load(
        REFERENCE_DIR / "oof_50_50_blend.npy", allow_pickle=False
    )
    reference_test = np.load(
        REFERENCE_DIR / "test_50_50_blend.npy", allow_pickle=False
    )
    folds = []
    for fold_number in range(1, 6):
        with np.load(REFERENCE_DIR / f"fold_{fold_number}_indices.npz", allow_pickle=False) as archive:
            folds.append((archive["fit_indices"].copy(), archive["valid_indices"].copy()))
    return arrays, reference_test, folds


def prediction_comparison(
    current: np.ndarray, reference: np.ndarray
) -> dict[str, float]:
    current = np.asarray(current, dtype=np.float64)
    reference = np.asarray(reference, dtype=np.float64)
    if current.shape != reference.shape or current.ndim != 1:
        raise ValueError("Reference and current prediction shapes differ.")
    delta = current - reference
    correlation = float(np.corrcoef(current, reference)[0, 1])
    return {
        "pearson": correlation,
        "mae": float(np.mean(np.abs(delta))),
        "rmse": float(np.sqrt(np.mean(np.square(delta)))),
        "maximum_absolute_difference": float(np.max(np.abs(delta))),
        "current_mean": float(current.mean()),
        "reference_mean": float(reference.mean()),
        "current_std": float(current.std()),
        "reference_std": float(reference.std()),
    }


def run_pipeline(
    data_dir: Path = DEFAULT_DATA_DIR,
    mode: str = "full",
    run_id: str | None = None,
) -> dict[str, Any]:
    if mode not in {"full", "reproduce"}:
        raise ValueError("Mode must be 'full' or 'reproduce'.")
    data_dir = data_dir.resolve()
    if run_id is None:
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_phase2_complete")
    if not run_id.replace("_", "").replace("-", "").isalnum():
        raise ValueError("Run ID may contain only letters, digits, underscore, and hyphen.")
    run_dir = PACKAGE_DIR / "runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    for relative in ("models", "predictions", "submissions", "reports"):
        (run_dir / relative).mkdir()
    started_at = datetime.now(timezone.utc).isoformat()
    timer = time.perf_counter()
    source_files = sorted((PACKAGE_DIR / "source").glob("*.py"))
    config_files = sorted((PACKAGE_DIR / "config").glob("*.json"))
    authoritative_sources = [
        FROZEN_DIR / "SOURCE" / "project" / "source" / "run_independent_baseline.py",
        FROZEN_DIR
        / "SOURCE"
        / "project"
        / "experiments"
        / "phase_01_fee_product_features"
        / "run_experiment.py",
        FROZEN_DIR
        / "SOURCE"
        / "project"
        / "experiments"
        / "phase_02_fold_safe_key_encodings"
        / "run_experiment.py",
    ]
    run_state: dict[str, Any] = {
        "run_id": run_id,
        "mode": mode,
        "status": "RUNNING",
        "started_at_utc": started_at,
        "command": [sys.executable, *sys.argv],
        "working_directory": str(Path.cwd()),
        "git": git_evidence(),
        "software_versions": package_versions(),
        "source_hashes": file_inventory(source_files, PACKAGE_DIR),
        "authoritative_source_hashes": file_inventory(
            authoritative_sources, FROZEN_DIR / "SOURCE" / "project"
        ),
        "config_hashes": file_inventory(config_files, PACKAGE_DIR),
    }
    try:
        model_config = read_json(PACKAGE_DIR / "config" / "model_config.json")
        feature_config = read_json(PACKAGE_DIR / "config" / "feature_config.json")
        ensemble_config = read_json(PACKAGE_DIR / "config" / "ensemble_config.json")
        expected_hashes = {
            name: value["sha256"]
            for name, value in read_json(FROZEN_DIR / "HASHES" / "INPUT_HASHES.json")[
                "inputs"
            ].items()
        }
        reference_meta = verify_frozen_reference(data_dir)
        gpu_before = gpu_snapshot()
        if model_config["catboost"]["task_type"] == "GPU":
            gpu_preflight = require_gpu_preflight()
        else:
            raise ValueError("Frozen Phase 2 requires CatBoost GPU training.")

        train_raw, test_raw, sample, labels, input_metadata = load_inputs(
            data_dir, expected_hashes
        )
        if feature_config["fee_decoder"]["target_used"] is not False:
            raise ValueError("Fee feature configuration must be label-free.")
        if feature_config["fee_decoder"]["frequency_counts_use_train_and_test_covariates"] is not True:
            raise ValueError("Fee-frequency configuration differs from selected Phase 2.")
        expected_keys = [
            (item["name"], item["source_feature"], float(item["smoothing"]))
            for item in feature_config["target_encoding"]["keys"]
        ]
        if expected_keys != list(KEY_SPECS):
            raise ValueError("Target-encoding configuration differs from source implementation.")
        if tuple(feature_config["categorical_columns"]) != CATEGORICAL_COLUMNS:
            raise ValueError("Categorical feature configuration differs from source implementation.")
        if float(ensemble_config["lightgbm_weight"]) != 0.5 or float(
            ensemble_config["catboost_weight"]
        ) != 0.5:
            raise ValueError("The frozen Phase 2 ensemble is a 50/50 probability blend.")

        base_train = build_rowwise_features(train_raw.drop(columns=[TARGET_COLUMN]))
        base_test = build_rowwise_features(test_raw)
        fee_train, fee_test, feature_diagnostics = build_decoded_loan_features(
            train_raw,
            test_raw,
            pooled_frequency_reference=True,
        )
        base_train = pd.concat([base_train, fee_train], axis=1)
        base_test = pd.concat([base_test, fee_test], axis=1)
        keys_by_name = key_arrays(fee_train, fee_test)
        folds = build_folds(
            labels,
            n_splits=int(model_config["outer_cv"]["n_splits"]),
            seed=int(model_config["outer_cv"]["seed"]),
        )
        reference_oof, reference_test, reference_folds = load_reference_arrays()
        fold_identity = True
        for current, reference in zip(folds, reference_folds):
            if not (
                np.array_equal(current[0], reference[0])
                and np.array_equal(current[1], reference[1])
            ):
                fold_identity = False
                break
        if not fold_identity:
            raise ValueError("New fold assignments do not exactly match frozen Phase 2.")
        n_train = len(train_raw)
        test_indices = np.arange(n_train, n_train + len(test_raw), dtype=np.int64)
        fold_ids = fold_id_vector(labels, folds)
        np.savez_compressed(
            run_dir / "fold_assignments.npz",
            fold_id=fold_ids,
            **{
                f"fold_{index}_fit": fit.astype(np.int32)
                for index, (fit, _) in enumerate(folds, 1)
            },
            **{
                f"fold_{index}_valid": valid.astype(np.int32)
                for index, (_, valid) in enumerate(folds, 1)
            },
        )

        oof = {
            name: np.full(n_train, np.nan, dtype=np.float32)
            for name in ("lightgbm", "catboost")
        }
        fold_test_predictions: dict[str, list[np.ndarray]] = {
            name: [] for name in ("lightgbm", "catboost")
        }
        fold_records = []
        feature_schema: dict[str, Any] | None = None
        for fold_number, (fit_indices, valid_indices) in enumerate(folds, 1):
            fold_started = time.perf_counter()
            train_te = nested_training_encodings(keys_by_name, labels, fit_indices)
            valid_te = apply_training_encodings(
                keys_by_name, labels, fit_indices, valid_indices
            )
            test_te = apply_training_encodings(
                keys_by_name, labels, fit_indices, test_indices
            )
            if np.intersect1d(fit_indices, valid_indices).size:
                raise RuntimeError("Outer target-encoding fit/validation indices overlap.")
            if not train_te.index.equals(pd.RangeIndex(len(fit_indices))):
                raise RuntimeError("Nested encodings are not aligned to outer fit indices.")
            X_train_fold = base_train.copy()
            X_train_fold.loc[fit_indices, train_te.columns] = train_te.to_numpy()
            X_train_fold.loc[valid_indices, valid_te.columns] = valid_te.to_numpy()
            X_test_fold = pd.concat([base_test, test_te], axis=1)
            if np.isinf(
                X_train_fold.select_dtypes(include=[np.number]).to_numpy(
                    dtype=np.float64, copy=False
                )
            ).any():
                raise ValueError("Fold feature matrix contains infinite values.")
            X_lgb_train, X_lgb_test = make_lightgbm_frames(X_train_fold, X_test_fold)
            if feature_schema is None:
                feature_schema = {
                    "catboost_features": [
                        {"name": name, "dtype": str(dtype)}
                        for name, dtype in X_train_fold.dtypes.items()
                    ],
                    "lightgbm_features": [
                        {"name": name, "dtype": str(dtype)}
                        for name, dtype in X_lgb_train.dtypes.items()
                    ],
                }
                write_json(run_dir / "feature_schema.json", feature_schema)

            lgb_settings = dict(model_config["lightgbm"])
            lgb_settings["num_threads"] = min(
                int(model_config["cpu_threads"]), os.cpu_count() or 1
            )
            cat_settings = dict(model_config["catboost"])
            cat_settings["num_threads"] = min(
                int(model_config["cpu_threads"]), os.cpu_count() or 1
            )
            cat_settings["categorical_columns"] = list(CATEGORICAL_COLUMNS)
            lgb_result = train_lightgbm_fold(
                X_lgb_train,
                labels,
                X_lgb_test,
                fit_indices,
                valid_indices,
                fold_number,
                lgb_settings,
                int(model_config["outer_cv"]["seed"]) + fold_number,
                run_dir / "models" / "lightgbm" / f"fold_{fold_number}.txt",
            )
            cat_result = train_catboost_fold(
                X_train_fold,
                labels,
                X_test_fold,
                fit_indices,
                valid_indices,
                fold_number,
                cat_settings,
                int(model_config["outer_cv"]["seed"]) + fold_number,
                run_dir / "models" / "catboost" / f"fold_{fold_number}.cbm",
            )
            for name, result in (("lightgbm", lgb_result), ("catboost", cat_result)):
                oof[name][valid_indices] = result["valid_prediction"]
                fold_test_predictions[name].append(result["test_prediction"])
            fold_records.append(
                {
                    "fold": fold_number,
                    "fit_rows": int(len(fit_indices)),
                    "valid_rows": int(len(valid_indices)),
                    "fit_validation_disjoint": True,
                    "nested_te_rows": int(len(train_te)),
                    "lightgbm": {
                        "auc": lgb_result["auc"],
                        "best_iteration": lgb_result["best_iteration"],
                        "device": lgb_result["device"],
                    },
                    "catboost": {
                        "auc": cat_result["auc"],
                        "best_iteration": cat_result["best_iteration"],
                        "device": cat_result["device"],
                        "gpu_telemetry": cat_result["gpu_telemetry"],
                    },
                    "elapsed_seconds": time.perf_counter() - fold_started,
                }
            )
            pd.concat(
                [lgb_result["importance"]], ignore_index=True
            ).to_csv(run_dir / f"lightgbm_importance_fold_{fold_number}.csv", index=False)
            cat_result["importance"].to_csv(
                run_dir / f"catboost_importance_fold_{fold_number}.csv", index=False
            )
            np.savez_compressed(
                run_dir / f"fold_{fold_number}_indices.npz",
                fit_indices=fit_indices.astype(np.int32),
                valid_indices=valid_indices.astype(np.int32),
            )
            print(
                f"fold={fold_number}/5 lgb_auc={lgb_result['auc']:.8f} "
                f"cat_auc={cat_result['auc']:.8f} "
                f"best_iter={lgb_result['best_iteration']}/"
                f"{cat_result['best_iteration']}",
                flush=True,
            )

        if any(not np.isfinite(values).all() for values in oof.values()):
            raise ValueError("At least one OOF prediction is missing or non-finite.")
        test_predictions = {
            name: mean_fold_predictions(values)
            for name, values in fold_test_predictions.items()
        }
        blend_oof = combine_phase2_predictions(oof["lightgbm"], oof["catboost"])
        blend_test = combine_phase2_predictions(
            test_predictions["lightgbm"], test_predictions["catboost"]
        )
        for name in ("lightgbm", "catboost"):
            np.save(run_dir / "predictions" / f"oof_{name}.npy", oof[name])
            np.save(run_dir / "predictions" / f"test_{name}.npy", test_predictions[name])
        np.save(run_dir / "predictions" / "oof_phase2_blend.npy", blend_oof)
        np.save(run_dir / "predictions" / "test_phase2_blend.npy", blend_test)
        importance_frames = {
            name: pd.concat(
                [
                    pd.read_csv(run_dir / f"{name}_importance_fold_{fold}.csv")
                    for fold in range(1, 6)
                ],
                ignore_index=True,
            )
            for name in ("lightgbm", "catboost")
        }
        for name, frame in importance_frames.items():
            frame.to_csv(run_dir / f"{name}_feature_importance.csv", index=False)
        candidate_path = run_dir / "submissions" / "submission_phase2_complete.csv"
        submission_check = build_submission(test_raw, sample, blend_test, candidate_path)
        candidate_sha = sha256_file(candidate_path)

        reference_frame = pd.read_csv(REFERENCE_CANDIDATE)
        candidate_frame = pd.read_csv(candidate_path)
        reference_submission_check = validate_submission(
            reference_frame, test_raw, sample
        )
        current_auc = float(roc_auc_score(labels, blend_oof))
        current_fold_auc = [
            float(roc_auc_score(labels[valid], blend_oof[valid]))
            for _, valid in folds
        ]
        reference_fold_auc = [
            float(roc_auc_score(labels[valid], reference_oof["blend"][valid]))
            for _, valid in folds
        ]
        fold_auc_deltas = [
            current - reference
            for current, reference in zip(current_fold_auc, reference_fold_auc)
        ]
        oof_compare = prediction_comparison(blend_oof, reference_oof["blend"])
        test_compare = prediction_comparison(
            blend_test, reference_test.astype(np.float64)
        )
        candidate_values = candidate_frame[TARGET_COLUMN].to_numpy(dtype=np.float64)
        frozen_values = reference_frame[TARGET_COLUMN].to_numpy(dtype=np.float64)
        candidate_csv_compare = prediction_comparison(candidate_values, frozen_values)
        validation_limits = {
            "maximum_absolute_global_oof_auc_delta": 0.001,
            "maximum_absolute_per_fold_auc_delta": 0.003,
            "minimum_oof_blend_pearson": 0.99,
            "minimum_test_blend_pearson": 0.99,
            "minimum_submission_pearson": 0.99,
        }
        reproduction_checks = {
            "fold_assignments_exact": fold_identity,
            "global_oof_auc_within_tolerance": abs(
                current_auc - reference_meta["candidate_oof_auc"]
            )
            <= validation_limits["maximum_absolute_global_oof_auc_delta"],
            "per_fold_auc_within_tolerance": max(
                abs(value) for value in fold_auc_deltas
            )
            <= validation_limits["maximum_absolute_per_fold_auc_delta"],
            "oof_prediction_correlation_within_tolerance": oof_compare["pearson"]
            >= validation_limits["minimum_oof_blend_pearson"],
            "test_prediction_correlation_within_tolerance": test_compare["pearson"]
            >= validation_limits["minimum_test_blend_pearson"],
            "submission_correlation_within_tolerance": candidate_csv_compare["pearson"]
            >= validation_limits["minimum_submission_pearson"],
        }
        reproduction_pass = all(reproduction_checks.values())
        if not reproduction_pass:
            raise RuntimeError(
                "Reproduction validation failed; submission was not promoted. "
                + json.dumps(reproduction_checks, sort_keys=True)
            )
        promotion_path = None
        if mode == "full":
            promotion_path = PACKAGE_DIR / "submissions" / run_id / candidate_path.name
            if promotion_path.exists():
                raise FileExistsError(f"Refusing to overwrite submission: {promotion_path}")
            promotion_path.parent.mkdir(parents=True, exist_ok=False)
            shutil.copy2(candidate_path, promotion_path)
            if sha256_file(promotion_path) != candidate_sha:
                raise IOError("Promoted submission copy failed SHA-256 verification.")

        gpu_after = gpu_snapshot()
        report = {
            "run_id": run_id,
            "status": "PASS",
            "mode": mode,
            "classification": "INDEPENDENT_RAW_DATA_PHASE2_PIPELINE",
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "elapsed_seconds": time.perf_counter() - timer,
            "input_validation": input_metadata,
            "feature_diagnostics": feature_diagnostics,
            "feature_schema": feature_schema,
            "target_encoding": {
                "key_specs": [
                    {"name": name, "source_feature": column, "smoothing": smoothing}
                    for name, column, smoothing in KEY_SPECS
                ],
                "smoothing_formula": "(sum(target) + smoothing * fit_prior) / (count + smoothing)",
                "inner_folds": INNER_SPLITS,
                "inner_seed": INNER_SEED,
                "training_rows_inner_oof": True,
                "outer_validation_encoded_from_outer_fit_only": True,
                "test_encoded_from_outer_fit_only": True,
                "validation_labels_excluded_from_all_fold_encodings": True,
                "self_target_excluded_by_disjoint_fit_apply_indices": True,
            },
            "folds": {
                "outer_count": len(folds),
                "outer_seed": int(model_config["outer_cv"]["seed"]),
                "fold_assignments_match_frozen": fold_identity,
                "fold_records": fold_records,
            },
            "model_configuration": model_config,
            "models": {
                "lightgbm_device": "CPU",
                "catboost_device": "GPU:0",
                "lightgbm_fold_auc": [row["lightgbm"]["auc"] for row in fold_records],
                "catboost_fold_auc": [row["catboost"]["auc"] for row in fold_records],
                "lightgbm_best_iterations": [
                    row["lightgbm"]["best_iteration"] for row in fold_records
                ],
                "catboost_best_iterations": [
                    row["catboost"]["best_iteration"] for row in fold_records
                ],
            },
            "ensemble": {
                "formula": "0.5 * LightGBM raw probability + 0.5 * CatBoost raw probability",
                "post_processing": "none",
                "oof_auc": current_auc,
                "per_fold_auc": current_fold_auc,
                "reference_oof_auc": reference_meta["candidate_oof_auc"],
                "oof_auc_delta": current_auc - reference_meta["candidate_oof_auc"],
            },
            "reproduction_validation": {
                "status": "PASS",
                "reference_candidate_sha256": EXPECTED_CANDIDATE_SHA256,
                "limits": validation_limits,
                "checks": reproduction_checks,
                "fold_auc_delta": fold_auc_deltas,
                "oof_prediction_comparison": oof_compare,
                "test_prediction_comparison": test_compare,
                "submission_comparison": candidate_csv_compare,
                "frozen_submission_checks": reference_submission_check,
            },
            "submission": {
                "run_path": str(candidate_path.resolve()),
                "sha256": candidate_sha,
                "verification": submission_check,
                "promoted_path": str(promotion_path.resolve()) if promotion_path else None,
                "promoted_sha256": (
                    sha256_file(promotion_path) if promotion_path else None
                ),
            },
            "gpu": {
                "before": gpu_before,
                "preflight": gpu_preflight,
                "after": gpu_after,
                "catboost_fold_telemetry": [
                    row["catboost"]["gpu_telemetry"] for row in fold_records
                ],
            },
            "reference": reference_meta,
            "software_versions": package_versions(),
            "authoritative_source_hashes": run_state[
                "authoritative_source_hashes"
            ],
            "git": git_evidence(),
            "official_competition_score": "NOT_EVALUATED",
        }
        report_path = run_dir / "reports" / "implementation_report.json"
        write_json(report_path, report)
        write_markdown_report(run_dir / "reports" / "implementation_report.md", report)
        write_json(PACKAGE_DIR / "IMPLEMENTATION_REPORT.json", report)
        write_markdown_report(PACKAGE_DIR / "IMPLEMENTATION_REPORT.md", report)
        run_state.update(
            {
                "status": "PASS",
                "completed_at_utc": report["completed_at_utc"],
                "elapsed_seconds": report["elapsed_seconds"],
                "inputs": input_metadata["input_hashes"],
                "frozen_reference": reference_meta,
                "effective_cpu_threads": min(
                    int(model_config["cpu_threads"]), os.cpu_count() or 1
                ),
                "outputs": None,
            }
        )
        write_json(run_dir / "run_manifest.json", run_state)
        outputs = {
            path.relative_to(run_dir).as_posix(): {
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
            for path in sorted(run_dir.rglob("*"))
            if path.is_file()
            and path.name not in {"sha256_manifest.json", "run_manifest.json"}
        }
        run_state["outputs"] = outputs
        write_json(run_dir / "run_manifest.json", run_state)
        manifest_files = {
            path.relative_to(run_dir).as_posix(): {
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
            for path in sorted(run_dir.rglob("*"))
            if path.is_file() and path.name != "sha256_manifest.json"
        }
        write_json(
            run_dir / "sha256_manifest.json",
            {
                "run_id": run_id,
                "algorithm": "SHA-256",
                "files": manifest_files,
                "manifest_excludes_itself": True,
            },
        )
        global_manifest = PACKAGE_DIR / "runs" / "manifests" / f"{run_id}.json"
        write_json(global_manifest, run_state)
        global_report = PACKAGE_DIR / "runs" / "reports" / f"{run_id}.md"
        global_report.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(run_dir / "reports" / "implementation_report.md", global_report)
        print(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False))
        return report
    except Exception as error:
        failure = {
            "run_id": run_id,
            "status": "FAILED",
            "failed_at_utc": datetime.now(timezone.utc).isoformat(),
            "error_type": type(error).__name__,
            "error": str(error),
            "traceback": traceback.format_exc(),
            "run_state": run_state,
        }
        write_json(run_dir / "failure_report.json", failure)
        raise


def write_markdown_report(path: Path, report: dict[str, Any]) -> None:
    ensemble = report["ensemble"]
    submission = report["submission"]
    reproduction = report["reproduction_validation"]
    lines = [
        "# EXP85 Phase 2 Complete Pipeline — Execution Report",
        "",
        f"- Status: **{report['status']}**",
        f"- Mode: `{report['mode']}`",
        f"- Run ID: `{report['run_id']}`",
        f"- Runtime seconds: `{report['elapsed_seconds']:.3f}`",
        f"- OOF AUC: `{ensemble['oof_auc']:.10f}`",
        f"- Frozen OOF AUC: `{ensemble['reference_oof_auc']:.10f}`",
        f"- OOF AUC delta: `{ensemble['oof_auc_delta']:+.10f}`",
        f"- Submission SHA-256: `{submission['sha256']}`",
        f"- Reproduction gate: **{reproduction['status']}**",
        f"- Official competition score: **NOT EVALUATED**",
        "",
        "## Per-fold validation",
        "",
        "| Fold | LightGBM AUC | CatBoost AUC | Blend AUC | Frozen blend AUC | Delta |",
        "|---:|---:|---:|---:|---:|---:|",
    ]
    for index, (row, current, reference) in enumerate(
        zip(
            report["folds"]["fold_records"],
            ensemble["per_fold_auc"],
            report["reference"]["candidate_oof_by_fold"],
        ),
        1,
    ):
        lines.append(
            f"| {index} | {row['lightgbm']['auc']:.8f} | "
            f"{row['catboost']['auc']:.8f} | {current:.8f} | "
            f"{reference:.8f} | {current - reference:+.8f} |"
        )
    lines.extend(
        [
            "",
            "## Device evidence",
            "",
            f"- LightGBM: `{report['models']['lightgbm_device']}`",
            f"- CatBoost: `{report['models']['catboost_device']}`",
            f"- GPU preflight: `{report['gpu']['preflight']['catboost_gpu_smoke_test']}`",
            f"- GPU telemetry samples: "
            f"`{sum(item['sample_count'] for item in report['gpu']['catboost_fold_telemetry'])}`",
            "",
            "## Reproduction comparison",
            "",
            f"- OOF blend correlation: `{reproduction['oof_prediction_comparison']['pearson']:.8f}`",
            f"- Test blend correlation: `{reproduction['test_prediction_comparison']['pearson']:.8f}`",
            f"- Submission correlation: `{reproduction['submission_comparison']['pearson']:.8f}`",
            f"- Promoted submission: `{submission['promoted_path']}`",
            "",
            "## Provenance",
            "",
            "- Raw training/test/sample inputs were hash-verified and no historical EXP85 predictions were used.",
            "- The frozen Phase 2 candidate and its saved OOF/test outputs were read-only comparison references.",
            "- Authoritative frozen baseline/Phase 1/Phase 2 source SHA-256 values are recorded in the JSON report and run manifest.",
            "- Full raw-data retraining is independent of historical-cache reconstruction.",
            "",
        ]
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines), encoding="utf-8", newline="\n")
'''
_write_embedded_source('source/pipeline.py', _EMBEDDED_SOURCE_TEXT, '30a15b3eee7f3b84f5bd8a7d19ddfdb246704b745360bae0edbf3d39aadbdeb0')
del _EMBEDDED_SOURCE_TEXT


### Embedded `scripts/run_full_pipeline.py`

In [ ]:
# Verbatim embedded source: scripts/run_full_pipeline.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from pipeline import DEFAULT_DATA_DIR, run_pipeline


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Train the frozen Phase 2 recipe from raw CSVs and verify its submission."
    )
    parser.add_argument("--data-dir", type=Path, default=DEFAULT_DATA_DIR)
    parser.add_argument("--run-id", help="Unique run name; never overwritten.")
    parser.add_argument("--mode", choices=("full", "reproduce"), default="full")
    args = parser.parse_args()
    report = run_pipeline(args.data_dir, args.mode, args.run_id)
    return 0 if report["status"] == "PASS" else 1


if __name__ == "__main__":
    raise SystemExit(main())
'''
_write_embedded_source('scripts/run_full_pipeline.py', _EMBEDDED_SOURCE_TEXT, '108a2a7f0735f089d31325cf2f799eb097c2e23bfd39f4150f341bb4c6729b07')
del _EMBEDDED_SOURCE_TEXT


### Embedded `scripts/reproduce_validation.py`

In [ ]:
# Verbatim embedded source: scripts/reproduce_validation.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from pipeline import DEFAULT_DATA_DIR, run_pipeline


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Retrain seed-42 Phase 2 and compare all folds/predictions with the frozen reference."
    )
    parser.add_argument("--data-dir", type=Path, default=DEFAULT_DATA_DIR)
    parser.add_argument("--run-id", help="Unique run name; never overwritten.")
    args = parser.parse_args()
    report = run_pipeline(args.data_dir, "reproduce", args.run_id)
    return 0 if report["reproduction_validation"]["status"] == "PASS" else 1


if __name__ == "__main__":
    raise SystemExit(main())
'''
_write_embedded_source('scripts/reproduce_validation.py', _EMBEDDED_SOURCE_TEXT, 'd1827019968d299e9af6f7b94c8b5bdb14c9b83d2d4f7561ae37d2715557a6bd')
del _EMBEDDED_SOURCE_TEXT


### Embedded `scripts/verify_submission.py`

In [ ]:
# Verbatim embedded source: scripts/verify_submission.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from submission_builder import validate_submission


def main() -> int:
    parser = argparse.ArgumentParser(description="Independently verify a Phase 2 submission CSV.")
    parser.add_argument("--submission", type=Path, required=True)
    parser.add_argument("--data-dir", type=Path, default=Path(r"D:\gci\inpuy"))
    args = parser.parse_args()
    test = pd.read_csv(args.data_dir / "test.csv")
    sample = pd.read_csv(args.data_dir / "sample_submission.csv")
    candidate = pd.read_csv(args.submission)
    result = validate_submission(candidate, test, sample)
    print(json.dumps(result, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''
_write_embedded_source('scripts/verify_submission.py', _EMBEDDED_SOURCE_TEXT, '7a4aada89a8f65a5f02c1908ba44b817d09cb77a37a696a4d75b46ead7bc63ee')
del _EMBEDDED_SOURCE_TEXT


### Embedded `scripts/verify_pipeline.py`

In [ ]:
# Verbatim embedded source: scripts/verify_pipeline.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from data_validation import sha256_file
from submission_builder import validate_submission


def verify_run(run_dir: Path, data_dir: Path) -> dict[str, object]:
    run_manifest_path = run_dir / "run_manifest.json"
    hash_manifest_path = run_dir / "sha256_manifest.json"
    if not run_manifest_path.is_file() or not hash_manifest_path.is_file():
        raise FileNotFoundError("Run or SHA-256 manifest is missing.")
    run_manifest = json.loads(run_manifest_path.read_text(encoding="utf-8"))
    hash_manifest = json.loads(hash_manifest_path.read_text(encoding="utf-8"))
    if run_manifest.get("status") != "PASS":
        raise ValueError(f"Pipeline status is not PASS: {run_manifest.get('status')}")
    expected = hash_manifest["files"]
    actual_paths = {
        path.relative_to(run_dir).as_posix()
        for path in run_dir.rglob("*")
        if path.is_file() and path.name != "sha256_manifest.json"
    }
    if actual_paths != set(expected):
        raise ValueError("Run directory file inventory differs from its manifest.")
    for relative, record in expected.items():
        path = run_dir / Path(relative)
        if path.stat().st_size != record["size_bytes"] or sha256_file(path) != record["sha256"]:
            raise ValueError(f"Run artifact hash mismatch: {relative}")
    report = json.loads(
        (run_dir / "reports" / "implementation_report.json").read_text(
            encoding="utf-8"
        )
    )
    if report["reproduction_validation"]["status"] != "PASS":
        raise ValueError("Frozen-reference reproduction gate did not pass.")
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    sample = pd.read_csv(data_dir / "sample_submission.csv")
    candidate_path = run_dir / "submissions" / "submission_phase2_complete.csv"
    candidate = pd.read_csv(candidate_path)
    submission_checks = validate_submission(candidate, test, sample)
    for name in ("lightgbm", "catboost", "phase2_blend"):
        for kind, expected_rows in (("oof", len(train)), ("test", len(test))):
            prediction_path = run_dir / "predictions" / f"{kind}_{name}.npy"
            if not prediction_path.is_file():
                raise FileNotFoundError(
                    f"Required {kind} prediction artifact is missing: {prediction_path}"
                )
            predictions = np.load(prediction_path, allow_pickle=False)
            if (
                predictions.shape != (expected_rows,)
                or not np.isfinite(predictions).all()
                or np.any((predictions < 0.0) | (predictions > 1.0))
            ):
                raise ValueError(f"Invalid {kind} predictions: {prediction_path.name}")
    promotion_path = report["submission"]["promoted_path"]
    if promotion_path:
        promoted = Path(promotion_path)
        if not promoted.is_file() or sha256_file(promoted) != sha256_file(candidate_path):
            raise ValueError("Promoted submission is missing or differs from verified run output.")
    return {
        "status": "PASS",
        "run_id": run_manifest["run_id"],
        "manifest_files": len(expected),
        "submission_sha256": sha256_file(candidate_path),
        "submission_checks": submission_checks,
        "promoted_path": promotion_path,
    }


def main() -> int:
    parser = argparse.ArgumentParser(description="Verify a completed pipeline run and submission.")
    parser.add_argument("--run-dir", type=Path, required=True)
    parser.add_argument("--data-dir", type=Path, default=Path(r"D:\gci\inpuy"))
    args = parser.parse_args()
    result = verify_run(args.run_dir.resolve(), args.data_dir.resolve())
    print(json.dumps(result, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''
_write_embedded_source('scripts/verify_pipeline.py', _EMBEDDED_SOURCE_TEXT, '8030c22ce5a68b1a874cfe8b361e8549b3d073c0ac048f7983829e59dc2b50e5')
del _EMBEDDED_SOURCE_TEXT


### Embedded `scripts/package_manifest.py`

In [ ]:
# Verbatim embedded source: scripts/package_manifest.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path
from typing import Any


PACKAGE_DIR = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(PACKAGE_DIR / "source"))

from data_validation import sha256_file


MANIFEST_NAME = "PACKAGE_SHA256_MANIFEST.json"


def inventory(package_dir: Path, manifest_path: Path) -> dict[str, dict[str, Any]]:
    files = {}
    for path in sorted(package_dir.rglob("*")):
        if (
            not path.is_file()
            or path == manifest_path
            or "__pycache__" in path.parts
            or path.suffix in {".pyc", ".pyo"}
        ):
            continue
        files[path.relative_to(package_dir).as_posix()] = {
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        }
    return files


def create_manifest(package_dir: Path) -> dict[str, Any]:
    package_dir = package_dir.resolve()
    manifest_path = package_dir / MANIFEST_NAME
    result = {
        "algorithm": "SHA-256",
        "manifest_excludes_itself": True,
        "files": inventory(package_dir, manifest_path),
    }
    manifest_path.write_text(
        json.dumps(result, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
        newline="\n",
    )
    return {"status": "PASS", "manifest": str(manifest_path), "file_count": len(result["files"])}


def verify_manifest(package_dir: Path) -> dict[str, Any]:
    package_dir = package_dir.resolve()
    manifest_path = package_dir / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Package SHA-256 manifest is missing: {manifest_path}")
    expected_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = expected_manifest.get("files")
    if not isinstance(expected, dict):
        raise ValueError("Package SHA-256 manifest has no valid file inventory.")
    actual = inventory(package_dir, manifest_path)
    missing = sorted(set(expected) - set(actual))
    extra = sorted(set(actual) - set(expected))
    mismatched = [
        path
        for path in sorted(set(expected) & set(actual))
        if expected[path] != actual[path]
    ]
    if missing or extra or mismatched:
        raise ValueError(
            json.dumps(
                {"missing": missing, "extra": extra, "mismatched": mismatched},
                sort_keys=True,
            )
        )
    return {
        "status": "PASS",
        "manifest": str(manifest_path),
        "file_count": len(expected),
    }


def main() -> int:
    parser = argparse.ArgumentParser(description="Create or verify the package SHA-256 inventory.")
    parser.add_argument("action", choices=("create", "verify"))
    parser.add_argument("--package-dir", type=Path, default=PACKAGE_DIR)
    args = parser.parse_args()
    result = (
        create_manifest(args.package_dir)
        if args.action == "create"
        else verify_manifest(args.package_dir)
    )
    print(json.dumps(result, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''
_write_embedded_source('scripts/package_manifest.py', _EMBEDDED_SOURCE_TEXT, 'f2d389857bf6a83eeb33adbc1c41fcd0cf3e7fee41917003c14c38f6d5e155ed')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_blending.py`

In [ ]:
# Verbatim embedded source: tests/test_blending.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import sys
import unittest
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from prediction_blender import combine_phase2_predictions, mean_fold_predictions


class BlendingTests(unittest.TestCase):
    def test_exact_half_probability_blend(self):
        lgb = np.asarray([0.2, 0.4], dtype=np.float32)
        cat = np.asarray([0.6, 0.8], dtype=np.float32)
        np.testing.assert_allclose(
            combine_phase2_predictions(lgb, cat), [0.4, 0.6], rtol=0.0, atol=2e-8
        )

    def test_average_fold_test_predictions_before_blend(self):
        result = mean_fold_predictions(
            [np.asarray([0.1, 0.5]), np.asarray([0.3, 0.9])]
        )
        np.testing.assert_array_equal(result, np.asarray([0.2, 0.7], dtype=np.float32))

    def test_rejects_shape_nonfinite_and_out_of_range_inputs(self):
        with self.assertRaisesRegex(ValueError, "shapes differ"):
            combine_phase2_predictions(np.asarray([0.2]), np.asarray([0.2, 0.3]))
        with self.assertRaisesRegex(ValueError, "finite"):
            combine_phase2_predictions(np.asarray([np.nan]), np.asarray([0.2]))
        with self.assertRaisesRegex(ValueError, "lie in"):
            combine_phase2_predictions(np.asarray([1.2]), np.asarray([0.2]))


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_blending.py', _EMBEDDED_SOURCE_TEXT, '874094e18701c5f929110488e518bf3bafa9b918789f77606043a4365ec5b9b1')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_data_validation.py`

In [ ]:
# Verbatim embedded source: tests/test_data_validation.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import sys
import unittest
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from data_validation import validate_frames


def frames():
    train = pd.DataFrame(
        {"SK_ID_CURR": [1, 2, 3, 4], "feature": [1.0, 2.0, 3.0, 4.0], "TARGET": [0, 1, 0, 1]}
    )
    test = pd.DataFrame({"SK_ID_CURR": [5, 6], "feature": [5.0, 6.0]})
    sample = pd.DataFrame({"SK_ID_CURR": [5, 6], "TARGET": [0, 0]})
    return train, test, sample


class DataValidationTests(unittest.TestCase):
    def test_valid_schema_target_and_id_order(self):
        labels, metadata = validate_frames(*frames())
        self.assertEqual(labels.tolist(), [0, 1, 0, 1])
        self.assertTrue(metadata["test_ids_equal_sample_order"])

    def test_rejects_missing_target_or_test_target(self):
        train, test, sample = frames()
        with self.assertRaisesRegex(ValueError, "TARGET"):
            validate_frames(train.drop(columns="TARGET"), test, sample)
        with self.assertRaisesRegex(ValueError, "exclude"):
            validate_frames(train, test.assign(TARGET=0), sample)

    def test_rejects_incompatible_predictor_order_and_sample_schema(self):
        train, test, sample = frames()
        with self.assertRaisesRegex(ValueError, "incompatible"):
            validate_frames(train, test[["feature", "SK_ID_CURR"]], sample)
        with self.assertRaisesRegex(ValueError, "sample schema"):
            validate_frames(train, test, sample[["TARGET", "SK_ID_CURR"]])

    def test_rejects_duplicate_ids_and_nonbinary_labels(self):
        train, test, sample = frames()
        with self.assertRaisesRegex(ValueError, "duplicated"):
            validate_frames(train, test.assign(SK_ID_CURR=[5, 5]), sample)
        train.loc[0, "TARGET"] = 2
        with self.assertRaisesRegex(ValueError, "binary"):
            validate_frames(train, test, sample)


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_data_validation.py', _EMBEDDED_SOURCE_TEXT, '70cea9892a191957d2f579c58318e0405b8f060ebaf66272a52e29884fe867df')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_fold_leakage.py`

In [ ]:
# Verbatim embedded source: tests/test_fold_leakage.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import sys
import unittest
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from folds import build_folds, fold_id_vector


class FoldLeakageTests(unittest.TestCase):
    def test_folds_are_reproducible_disjoint_and_cover_rows(self):
        labels = np.asarray([0, 1] * 100)
        first = build_folds(labels, 5, 42)
        second = build_folds(labels, 5, 42)
        self.assertEqual(len(first), 5)
        np.testing.assert_array_equal(fold_id_vector(labels, first), fold_id_vector(labels, second))
        for fit, valid in first:
            self.assertEqual(np.intersect1d(fit, valid).size, 0)
            self.assertEqual(len(fit) + len(valid), len(labels))
        self.assertTrue(np.all(fold_id_vector(labels, first) > 0))

    def test_different_seed_changes_fold_membership(self):
        labels = np.asarray([0, 1] * 100)
        first = build_folds(labels, 5, 42)
        second = build_folds(labels, 5, 43)
        self.assertFalse(all(np.array_equal(a[1], b[1]) for a, b in zip(first, second)))


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_fold_leakage.py', _EMBEDDED_SOURCE_TEXT, '5fc459e004bf60774678bb11da3ce59c7333ac8565e1739dd1bf7706ec42c3ba')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_manifest.py`

In [ ]:
# Verbatim embedded source: tests/test_manifest.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import hashlib
import json
import sys
import tempfile
import unittest
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "scripts"))

from verify_pipeline import verify_run


class ManifestTests(unittest.TestCase):
    def test_detects_modified_run_output_hash(self):
        with tempfile.TemporaryDirectory() as directory:
            run = Path(directory) / "run"
            (run / "reports").mkdir(parents=True)
            (run / "submissions").mkdir()
            (run / "predictions").mkdir()
            (run / "submissions" / "submission_phase2_complete.csv").write_text(
                "SK_ID_CURR,TARGET\n1,0.5\n", encoding="utf-8"
            )
            report = {
                "reproduction_validation": {"status": "PASS"},
                "submission": {"promoted_path": None},
            }
            (run / "reports" / "implementation_report.json").write_text(
                json.dumps(report), encoding="utf-8"
            )
            (run / "run_manifest.json").write_text(
                json.dumps({"status": "PASS", "run_id": "test"}), encoding="utf-8"
            )
            files = {}
            for path in run.rglob("*"):
                if path.is_file():
                    files[path.relative_to(run).as_posix()] = {
                        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                        "size_bytes": path.stat().st_size,
                    }
            (run / "sha256_manifest.json").write_text(
                json.dumps({"files": files}), encoding="utf-8"
            )
            input_dir = Path(directory) / "data"
            input_dir.mkdir()
            pd = __import__("pandas")
            pd.DataFrame({"SK_ID_CURR": [1], "feature": [0]}).to_csv(input_dir / "test.csv", index=False)
            pd.DataFrame({"SK_ID_CURR": [1], "TARGET": [0]}).to_csv(input_dir / "sample_submission.csv", index=False)
            (run / "submissions" / "submission_phase2_complete.csv").write_text(
                "SK_ID_CURR,TARGET\n1,0.6\n", encoding="utf-8"
            )
            with self.assertRaisesRegex(ValueError, "hash mismatch"):
                verify_run(run, input_dir)

    def test_requires_every_oof_and_test_prediction_file(self):
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            run = root / "run"
            for name in ("reports", "submissions", "predictions"):
                (run / name).mkdir(parents=True, exist_ok=True)
            data = root / "data"
            data.mkdir()
            pd.DataFrame({"SK_ID_CURR": [1, 2], "feature": [0, 1], "TARGET": [0, 1]}).to_csv(
                data / "train.csv", index=False
            )
            pd.DataFrame({"SK_ID_CURR": [3], "feature": [2]}).to_csv(
                data / "test.csv", index=False
            )
            pd.DataFrame({"SK_ID_CURR": [3], "TARGET": [0.5]}).to_csv(
                data / "sample_submission.csv", index=False
            )
            pd.DataFrame({"SK_ID_CURR": [3], "TARGET": [0.5]}).to_csv(
                run / "submissions" / "submission_phase2_complete.csv", index=False
            )
            (run / "reports" / "implementation_report.json").write_text(
                json.dumps(
                    {
                        "reproduction_validation": {"status": "PASS"},
                        "submission": {"promoted_path": None},
                    }
                ),
                encoding="utf-8",
            )
            (run / "run_manifest.json").write_text(
                json.dumps({"status": "PASS", "run_id": "test"}), encoding="utf-8"
            )
            for model in ("lightgbm", "catboost", "phase2_blend"):
                np.save(run / "predictions" / f"test_{model}.npy", np.array([0.5]))
                if model != "lightgbm":
                    np.save(
                        run / "predictions" / f"oof_{model}.npy",
                        np.array([0.2, 0.8]),
                    )
            files = {}
            for path in run.rglob("*"):
                if path.is_file():
                    files[path.relative_to(run).as_posix()] = {
                        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                        "size_bytes": path.stat().st_size,
                    }
            (run / "sha256_manifest.json").write_text(
                json.dumps({"files": files}), encoding="utf-8"
            )
            with self.assertRaisesRegex(FileNotFoundError, "Required oof prediction"):
                verify_run(run, data)


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_manifest.py', _EMBEDDED_SOURCE_TEXT, 'e0177c0929f8a3fbdb08120a61948ee9453f83ed356f669def1b3bfeccd53486')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_package_manifest.py`

In [ ]:
# Verbatim embedded source: tests/test_package_manifest.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import sys
import tempfile
import unittest
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "scripts"))

from package_manifest import create_manifest, verify_manifest


class PackageManifestTests(unittest.TestCase):
    def test_create_and_verify_package_manifest(self):
        with tempfile.TemporaryDirectory() as directory:
            package = Path(directory)
            (package / "source").mkdir()
            (package / "source" / "module.py").write_text("value = 1\n", encoding="utf-8")
            create_manifest(package)
            self.assertEqual(verify_manifest(package)["status"], "PASS")
            (package / "source" / "module.py").write_text("value = 2\n", encoding="utf-8")
            with self.assertRaisesRegex(ValueError, "mismatched"):
                verify_manifest(package)


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_package_manifest.py', _EMBEDDED_SOURCE_TEXT, '518e59bcc3ab66f90cd5f76761cb779a8120ee2654b4323a82a51429a37ec75c')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_phase1_features.py`

In [ ]:
# Verbatim embedded source: tests/test_phase1_features.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import sys
import unittest
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from phase1_features import build_decoded_loan_features
from preprocessing import CATEGORICAL_COLUMNS, build_rowwise_features, make_lightgbm_frames


def raw_rows(target: bool = False) -> pd.DataFrame:
    rows = []
    for index, (credit, annuity, ext) in enumerate(
        ((10000.0, 329.5, 0.3), (20000.0, 529.5, 0.5), (30000.0, 729.5, np.nan))
    ):
        row = {
            "SK_ID_CURR": 101 + index,
            "NAME_CONTRACT_TYPE": "Cash loans",
            "CODE_GENDER": "F" if index % 2 == 0 else "M",
            "NAME_INCOME_TYPE": "Working",
            "NAME_EDUCATION_TYPE": "Higher education",
            "NAME_FAMILY_STATUS": "Married",
            "NAME_HOUSING_TYPE": "House",
            "OCCUPATION_TYPE": "Laborers",
            "ORGANIZATION_TYPE": "Business",
            "CNT_CHILDREN": index,
            "AMT_INCOME_TOTAL": 90000.0 + index * 1000,
            "AMT_CREDIT": credit,
            "AMT_ANNUITY": annuity,
            "AMT_GOODS_PRICE": credit / 1.132,
            "CNT_FAM_MEMBERS": 2.0,
            "DAYS_BIRTH": -14000.0 - index,
            "DAYS_EMPLOYED": -1200.0,
            "DAYS_REGISTRATION": -3000.0,
            "DAYS_ID_PUBLISH": -5000.0,
            "DAYS_LAST_PHONE_CHANGE": -1000.0,
            "EXT_SOURCE_1": ext,
            "EXT_SOURCE_2": 0.5,
            "EXT_SOURCE_3": 0.6,
            "AMT_REQ_CREDIT_BUREAU_YEAR": 2.0,
        }
        if target:
            row["TARGET"] = index % 2
        rows.append(row)
    return pd.DataFrame(rows)


class Phase1FeatureTests(unittest.TestCase):
    def test_decoded_fee_features_and_product_tier_are_recovered(self):
        raw = raw_rows(target=True)
        fee_train, fee_test, info = build_decoded_loan_features(raw, raw.iloc[:1])
        self.assertEqual(info["fee_amount"], 129.5)
        self.assertTrue(fee_train["FEE_DECODED"].iloc[:2].eq(1).all())
        self.assertTrue(fee_train["PRODUCT_KEY_VALID"].eq(1).all())
        self.assertTrue(fee_train["PRODUCT_KEY_K"].eq(10).all())
        self.assertEqual(len(fee_test), 1)

    def test_features_are_label_independent_and_preserve_rows(self):
        raw = raw_rows(target=True)
        train = build_rowwise_features(raw.drop(columns="TARGET"))
        changed_target = raw.copy()
        changed_target["TARGET"] = 1 - changed_target["TARGET"]
        fee_a, test_a, _ = build_decoded_loan_features(raw, raw.iloc[:1])
        fee_b, test_b, _ = build_decoded_loan_features(changed_target, raw.iloc[:1])
        pd.testing.assert_frame_equal(train, build_rowwise_features(changed_target.drop(columns="TARGET")))
        pd.testing.assert_frame_equal(fee_a, fee_b)
        pd.testing.assert_frame_equal(test_a, test_b)
        self.assertEqual(len(train), len(raw))
        self.assertTrue(set(CATEGORICAL_COLUMNS).issubset(train.columns))

    def test_lightgbm_train_test_matrices_have_identical_schema(self):
        raw = raw_rows()
        train = build_rowwise_features(raw)
        test = build_rowwise_features(raw.iloc[:2])
        X_train, X_test = make_lightgbm_frames(train, test)
        self.assertEqual(list(X_train.columns), list(X_test.columns))
        self.assertFalse(np.isinf(X_train.to_numpy(dtype=np.float64)).any())


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_phase1_features.py', _EMBEDDED_SOURCE_TEXT, '6f07a79d4ecdb9db9701b53dee540eba4164fdde6efb05ec9635cb5a1b241af2')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_submission.py`

In [ ]:
# Verbatim embedded source: tests/test_submission.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import sys
import tempfile
import unittest
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from data_validation import sha256_file
from submission_builder import build_submission, validate_submission


class SubmissionTests(unittest.TestCase):
    def setUp(self):
        self.test = pd.DataFrame({"SK_ID_CURR": [10, 20, 30]})
        self.sample = pd.DataFrame({"SK_ID_CURR": [10, 20, 30], "TARGET": [0, 0, 0]})

    def test_build_writes_and_reloads_competition_schema(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "submission.csv"
            checks = build_submission(
                self.test, self.sample, np.asarray([0.2, 0.5, 0.8]), path
            )
            self.assertEqual(checks["status"], "PASS")
            self.assertEqual(list(pd.read_csv(path).columns), ["SK_ID_CURR", "TARGET"])
            self.assertEqual(len(sha256_file(path)), 64)

    def test_duplicate_ids_bad_order_and_invalid_probabilities_fail(self):
        invalid = pd.DataFrame(
            {"SK_ID_CURR": [10, 10, 30], "TARGET": [0.2, 0.5, 0.8]}
        )
        with self.assertRaisesRegex(ValueError, "duplicated"):
            validate_submission(invalid, self.test, self.sample)
        invalid = pd.DataFrame(
            {"SK_ID_CURR": [20, 10, 30], "TARGET": [0.2, 0.5, 0.8]}
        )
        with self.assertRaisesRegex(ValueError, "order"):
            validate_submission(invalid, self.test, self.sample)
        invalid = pd.DataFrame(
            {"SK_ID_CURR": [10, 20, 30], "TARGET": [0.2, np.inf, 0.8]}
        )
        with self.assertRaisesRegex(ValueError, "non-finite"):
            validate_submission(invalid, self.test, self.sample)


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_submission.py', _EMBEDDED_SOURCE_TEXT, 'bfc00679281b942a644299bd31fbc15f8fab1ddab2bb0bf92f85ce3b6f5d7f09')
del _EMBEDDED_SOURCE_TEXT


### Embedded `tests/test_target_encoding.py`

In [ ]:
# Verbatim embedded source: tests/test_target_encoding.py
_EMBEDDED_SOURCE_TEXT = r'''from __future__ import annotations

import sys
import unittest
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "source"))

from phase2_target_encoding import (
    KEY_SPECS,
    apply_training_encodings,
    encode_keys,
    nested_training_encodings,
)


def key_map(n: int) -> dict[str, np.ndarray]:
    base = np.asarray([f"key_{i % 7}" for i in range(n)], dtype=str)
    return {name: base.copy() for name, _, _ in KEY_SPECS}


class TargetEncodingTests(unittest.TestCase):
    def test_smoothing_formula_and_unseen_category_fallback(self):
        keys = np.asarray(["a", "a", "b", "b", "a", "new"])
        labels = np.asarray([0, 1, 0, 0, 1, 1], dtype=np.int8)
        encoded, counts = encode_keys(keys, labels, np.arange(4), np.asarray([4, 5]), "x", 2.0)
        prior = 0.25
        self.assertAlmostEqual(float(encoded[0]), (1 + 2 * prior) / 4)
        self.assertAlmostEqual(float(encoded[1]), prior)
        self.assertEqual(counts.tolist(), [2.0, 0.0])

    def test_apply_labels_cannot_change_training_mappings(self):
        keys = key_map(12)
        labels = np.asarray([0, 1] * 6, dtype=np.int8)
        fit = np.arange(8)
        apply = np.arange(8, 12)
        before = apply_training_encodings(keys, labels, fit, apply)
        changed = labels.copy()
        changed[apply] = 1 - changed[apply]
        after = apply_training_encodings(keys, changed, fit, apply)
        pd.testing.assert_frame_equal(before, after)

    def test_self_target_is_excluded_from_encoded_row(self):
        keys = np.asarray(["same"] * 8 + ["other"] * 4)
        labels = np.asarray([0, 1] * 6, dtype=np.int8)
        fit = np.asarray([0, 1, 2, 3, 4, 5])
        apply = np.asarray([6])
        first, _ = encode_keys(keys, labels, fit, apply, "test", 10.0)
        changed = labels.copy()
        changed[6] = 1 - changed[6]
        second, _ = encode_keys(keys, changed, fit, apply, "test", 10.0)
        np.testing.assert_array_equal(first, second)

    def test_outer_validation_labels_cannot_change_nested_training_encodings(self):
        labels = np.asarray([0, 1] * 50, dtype=np.int8)
        keys = key_map(len(labels))
        outer_fit = np.arange(80)
        before = nested_training_encodings(keys, labels, outer_fit)
        changed = labels.copy()
        changed[80:] = 1 - changed[80:]
        after = nested_training_encodings(keys, changed, outer_fit)
        pd.testing.assert_frame_equal(before, after)

    def test_fit_apply_overlap_is_rejected(self):
        labels = np.asarray([0, 1, 0, 1], dtype=np.int8)
        with self.assertRaisesRegex(ValueError, "overlap"):
            encode_keys(np.asarray(["a"] * 4), labels, np.asarray([0, 1]), np.asarray([1]), "x", 1.0)


if __name__ == "__main__":
    unittest.main()
'''
_write_embedded_source('tests/test_target_encoding.py', _EMBEDDED_SOURCE_TEXT, 'cde75436986c7b35a38fc594a0faabb2e9d953a6976331edbf28dd57afc4ff2c')
del _EMBEDDED_SOURCE_TEXT


## Embedded model and feature configurations

These are the exact JSON configurations used by the validated run, embedded directly so the notebook does not need the package `config/` files.

### Embedded `config/model_config.json`

In [ ]:
# Verbatim embedded configuration: model_config.json
_EMBEDDED_CONFIG_TEXT = r'''{
  "outer_cv": {
    "n_splits": 5,
    "shuffle": true,
    "seed": 42
  },
  "target_encoding_cv": {
    "n_splits": 5,
    "shuffle": true,
    "seed": 7
  },
  "cpu_threads": 8,
  "lightgbm": {
    "device": "CPU",
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 5.0,
    "max_bin": 127,
    "num_boost_round": 3000,
    "early_stopping_rounds": 150
  },
  "catboost": {
    "device": "GPU:0",
    "iterations": 2500,
    "depth": 7,
    "learning_rate": 0.045,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "l2_leaf_reg": 5.0,
    "random_strength": 0.5,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.8,
    "border_count": 128,
    "task_type": "GPU",
    "devices": "0",
    "early_stopping_rounds": 150
  }
}
'''
_write_embedded_source('config/model_config.json', _EMBEDDED_CONFIG_TEXT, 'c5c36b1287477c5d28725f33ce1ebb4ec804a444a110097cf56c9fc5cd28e4f9')
del _EMBEDDED_CONFIG_TEXT


### Embedded `config/feature_config.json`

In [ ]:
# Verbatim embedded configuration: feature_config.json
_EMBEDDED_CONFIG_TEXT = r'''{
  "id_column": "SK_ID_CURR",
  "target_column": "TARGET",
  "categorical_columns": [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE"
  ],
  "fee_decoder": {
    "fee_amount": 129.5,
    "product_rate": 0.0132,
    "frequency_counts_use_train_and_test_covariates": true,
    "target_used": false
  },
  "target_encoding": {
    "missing_key": "__MISSING_KEY__",
    "keys": [
      {
        "name": "FEE_Q3",
        "source_feature": "ANNUITY_CREDIT_RATIO_DECODED_ROUND_3",
        "round_digits": 3,
        "smoothing": 100.0
      },
      {
        "name": "FEE_Q4",
        "source_feature": "ANNUITY_CREDIT_RATIO_DECODED_ROUND_4",
        "round_digits": 4,
        "smoothing": 30.0
      },
      {
        "name": "FEE_Q5",
        "source_feature": "ANNUITY_CREDIT_RATIO_DECODED_ROUND_5",
        "round_digits": 5,
        "smoothing": 30.0
      },
      {
        "name": "PRODUCT_K",
        "source_feature": "PRODUCT_KEY_K",
        "round_digits": 0,
        "smoothing": 50.0
      }
    ],
    "formula": "(key_target_sum + smoothing * fit_prior) / (key_count + smoothing)"
  }
}
'''
_write_embedded_source('config/feature_config.json', _EMBEDDED_CONFIG_TEXT, '22929e36b480bb6791e707e1db33dcac65a199045c7075c8244fca4c7e0c0988')
del _EMBEDDED_CONFIG_TEXT


### Embedded `config/ensemble_config.json`

In [ ]:
# Verbatim embedded configuration: ensemble_config.json
_EMBEDDED_CONFIG_TEXT = r'''{
  "model_predictions": "raw binary probabilities",
  "lightgbm_weight": 0.5,
  "catboost_weight": 0.5,
  "formula": "0.5 * LightGBM_probability + 0.5 * CatBoost_probability",
  "test_prediction_aggregation": "mean of five fold model probabilities, then cast to float32",
  "oof_prediction_aggregation": "arithmetic mean after casting each model OOF vector to float64",
  "post_processing": "none",
  "output_decimal_places": 10
}
'''
_write_embedded_source('config/ensemble_config.json', _EMBEDDED_CONFIG_TEXT, 'fd04e39dde624c11a063448fbe94a9d5a6edb5e4f29e6be1adfdc07dd192e896')
del _EMBEDDED_CONFIG_TEXT


## Embedded frozen Phase 2 evidence

The next cell embeds a compressed, hash-checked copy of the frozen OOF/test prediction arrays, exact fold indices, manifests, baseline candidate, input hash record, and authoritative source files. It restores them into this notebook's temporary runtime so the original reproduction checks still work without external repository files.

In [ ]:
# Compressed frozen-reference bundle embedded in this notebook.
_EMBEDDED_EVIDENCE_B64 = (
    '...'
    'FgAWAJALAABUfEAAAAA='
)
_restore_embedded_evidence(_EMBEDDED_EVIDENCE_B64, '77d22c160d33eb746a655d2630cbcc411ad1454e930b3b48653c1d02a554eaf1')
del _EMBEDDED_EVIDENCE_B64


## Load the embedded implementation

Import the restored modules from this runtime only. No project Python file needs to be uploaded separately.

In [ ]:
sys.path[:0] = [str(PACKAGE_DIR / "source"), str(PACKAGE_DIR / "scripts")]

from data_validation import load_inputs, sha256_file
from folds import build_folds
from phase1_features import build_decoded_loan_features
from phase2_target_encoding import (
    KEY_SPECS,
    apply_training_encodings,
    key_arrays,
    nested_training_encodings,
)
from prediction_blender import combine_phase2_predictions, mean_fold_predictions
from preprocessing import build_rowwise_features, make_lightgbm_frames
from submission_builder import build_submission, validate_submission
from train_catboost import train_catboost_fold
from train_lightgbm import train_lightgbm_fold
from pipeline import load_reference_arrays, run_pipeline, verify_frozen_reference
from verify_pipeline import verify_run

MODEL_CONFIG = json.loads((PACKAGE_DIR / "config" / "model_config.json").read_text(encoding="utf-8"))
FEATURE_CONFIG = json.loads((PACKAGE_DIR / "config" / "feature_config.json").read_text(encoding="utf-8"))
ENSEMBLE_CONFIG = json.loads((PACKAGE_DIR / "config" / "ensemble_config.json").read_text(encoding="utf-8"))

print(f"Embedded source modules loaded: {len(list((PACKAGE_DIR / 'source').glob('*.py')))}")
print("Frozen evidence present:", FROZEN_DIR.is_dir())

## 1. Competition data loading and validation

Find the exact raw inputs and validate their SHA-256, schema, binary target, ID uniqueness, and test/sample ID order. If automatic discovery finds no exact input set, set `PHASE2_DATA_DIR` to the directory containing the three CSV files before running this cell.

In [ ]:
def _find_phase2_data_dir():
    input_manifest = json.loads(
        (FROZEN_DIR / "HASHES" / "INPUT_HASHES.json").read_text(encoding="utf-8")
    )
    expected = {name: record["sha256"] for name, record in input_manifest["inputs"].items()}
    candidates = []
    configured = os.environ.get("PHASE2_DATA_DIR")
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.extend([
        Path(r"D:\\gci\\inpuy"),
        Path.cwd(),
        Path.cwd() / "input",
        Path.cwd() / "data",
    ])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        candidates.extend(path.parent for path in kaggle_input.rglob("train.csv"))
    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except OSError:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if not all((candidate / name).is_file() for name in expected):
            continue
        if all(sha256_file(candidate / name) == digest for name, digest in expected.items()):
            return candidate
    raise FileNotFoundError(
        "Could not locate train.csv, test.csv, and sample_submission.csv with the "
        "validated Phase 2 SHA-256 hashes. Set PHASE2_DATA_DIR to the correct folder."
    )

DATA_DIR = _find_phase2_data_dir()
INPUT_MANIFEST = json.loads(
    (FROZEN_DIR / "HASHES" / "INPUT_HASHES.json").read_text(encoding="utf-8")
)
EXPECTED_INPUT_HASHES = {
    name: record["sha256"] for name, record in INPUT_MANIFEST["inputs"].items()
}
train_raw, test_raw, sample_submission, labels, input_checks = load_inputs(
    DATA_DIR, EXPECTED_INPUT_HASHES
)
print(json.dumps({
    "data_dir": str(DATA_DIR),
    "train_rows": input_checks["train_rows"],
    "test_rows": input_checks["test_rows"],
    "sample_rows": input_checks["sample_rows"],
    "class_counts": input_checks["class_counts"],
    "ids_unique": input_checks["ids_unique"],
    "test_ids_equal_sample_order": input_checks["test_ids_equal_sample_order"],
    "input_sha256": input_checks["input_hashes"],
}, indent=2))

## 2. Preprocessing

The full run uses the copied, unchanged row-wise feature builder and LightGBM categorical conversion. These smoke checks exercise their actual functions without fitting a model; full train/test feature generation occurs once inside the final pipeline run.

In [ ]:
_smoke_train = build_rowwise_features(train_raw.drop(columns=["TARGET"]).head(32))
_smoke_test = build_rowwise_features(test_raw.head(8))
assert list(_smoke_train.columns) == list(_smoke_test.columns)
_smoke_lgb_train, _smoke_lgb_test = make_lightgbm_frames(_smoke_train, _smoke_test)
assert list(_smoke_lgb_train.columns) == list(_smoke_lgb_test.columns)
print({
    "rowwise_features": _smoke_train.shape[1],
    "lightgbm_features": _smoke_lgb_train.shape[1],
    "train_test_schema_match": True,
})
del _smoke_train, _smoke_test, _smoke_lgb_train, _smoke_lgb_test

## 3. Phase 1 fee/product feature engineering

The unchanged Phase 1 implementation derives the 129.5 fee and 0.0132 product tier from raw loan fields. Fee-frequency counts pool train/test covariates without labels. The full pipeline performs this transformation once over all rows during training.

In [ ]:
assert FEATURE_CONFIG["fee_decoder"]["fee_amount"] == 129.5
assert FEATURE_CONFIG["fee_decoder"]["product_rate"] == 0.0132
assert FEATURE_CONFIG["fee_decoder"]["target_used"] is False
print("Phase 1 implementation:", build_decoded_loan_features.__module__)
print(json.dumps(FEATURE_CONFIG["fee_decoder"], indent=2))

## 4. Phase 2 nested target encoding

The four target-encoded keys and smoothing values are read from the validated config. Inner-training rows receive inner-fold OOF encodings; validation and test encodings use only outer-fit labels.

In [ ]:
configured_keys = [
    (item["name"], item["source_feature"], float(item["smoothing"]))
    for item in FEATURE_CONFIG["target_encoding"]["keys"]
]
assert configured_keys == list(KEY_SPECS)
print("Key specs:", KEY_SPECS)
print("Nested target encoding source:", nested_training_encodings.__module__)

## 5. Five-fold cross-validation

Create the exact seed-42 stratified folds and compare their indices to the frozen five-fold evidence. The full pipeline repeats this fail-closed check before model fitting.

In [ ]:
folds = build_folds(
    labels,
    n_splits=int(MODEL_CONFIG["outer_cv"]["n_splits"]),
    seed=int(MODEL_CONFIG["outer_cv"]["seed"]),
)
_, _, frozen_folds = load_reference_arrays()
assert len(folds) == len(frozen_folds) == 5
assert all(
    (fit == frozen_fit).all() and (valid == frozen_valid).all()
    for (fit, valid), (frozen_fit, frozen_valid) in zip(folds, frozen_folds)
)
print("Five folds exactly match frozen Phase 2 assignments.")

## 6. LightGBM CPU training

The existing CPU LightGBM training function, including archived parameters, fold-specific seed, and early stopping, is embedded and used by the final run. This cell only displays the selected configuration.

In [ ]:
assert callable(train_lightgbm_fold)
print(json.dumps(MODEL_CONFIG["lightgbm"], indent=2))

## 7. CatBoost GPU training

The original CatBoost trainer uses GPU 0 and fails closed if the validated RTX 4050 preflight fails; there is no silent CPU fallback. Full fold telemetry is retained in the run report.

In [ ]:
assert callable(train_catboost_fold)
print(json.dumps(MODEL_CONFIG["catboost"], indent=2))

## 8. OOF prediction generation

The full pipeline assigns each validation prediction to its original training-row position, verifies complete finite OOF vectors, and saves component and blend predictions. No fit is performed in this descriptive cell.

In [ ]:
assert callable(mean_fold_predictions)
print("OOF implementation is embedded in:", run_pipeline.__module__)
print("Training rows for OOF predictions:", len(labels))

## 9. 50/50 ensemble blending

The exact selected ensemble is the arithmetic mean of raw class-1 probabilities. There is no rank transform or post-processing.

In [ ]:
assert ENSEMBLE_CONFIG["lightgbm_weight"] == 0.5
assert ENSEMBLE_CONFIG["catboost_weight"] == 0.5
assert callable(combine_phase2_predictions)
print(ENSEMBLE_CONFIG["formula"])

## 10. Validation against frozen Phase 2

Verify the embedded frozen-release manifest, candidate hash, original input hashes, and comparison arrays before training.

In [ ]:
FROZEN_VALIDATION = verify_frozen_reference(DATA_DIR)
print(json.dumps(FROZEN_VALIDATION, indent=2))

## 11. Final test predictions

For each learner, five fold-specific test probability vectors are averaged; the resulting learner predictions are blended 50/50. The following full run creates and verifies these test predictions.

In [ ]:
print("Test IDs:", len(test_raw))
print("Expected final submission rows:", len(sample_submission))
assert len(test_raw) == len(sample_submission)
assert test_raw["SK_ID_CURR"].reset_index(drop=True).equals(
    sample_submission["SK_ID_CURR"].reset_index(drop=True)
)

## 12. Submission CSV generation and verification

Run the complete five-fold retraining and validation once. A unique run directory is created; promotion occurs only after the frozen reproduction gate passes. The finished CSV is also copied to a uniquely named file in the notebook working directory for convenient download.

In [ ]:
RUN_ID = f"notebook_phase2_{NOTEBOOK_RUN_ID}"
RUN_REPORT = run_pipeline(data_dir=DATA_DIR, mode="full", run_id=RUN_ID)
RUN_DIR = PACKAGE_DIR / "runs" / RUN_ID
assert RUN_REPORT["status"] == "PASS"
PROMOTED_SUBMISSION = Path(RUN_REPORT["submission"]["promoted_path"])
DOWNLOAD_SUBMISSION = Path.cwd() / f"submission_phase2_{RUN_ID}.csv"
if DOWNLOAD_SUBMISSION.exists():
    raise FileExistsError(f"Refusing to overwrite notebook submission: {DOWNLOAD_SUBMISSION}")
import shutil
shutil.copy2(PROMOTED_SUBMISSION, DOWNLOAD_SUBMISSION)
if sha256_file(DOWNLOAD_SUBMISSION) != RUN_REPORT["submission"]["sha256"]:
    raise IOError("Notebook-download submission copy failed SHA-256 verification.")
print(json.dumps({
    "run_id": RUN_ID,
    "run_directory": str(RUN_DIR),
    "submission_path": str(DOWNLOAD_SUBMISSION),
    "submission_sha256": RUN_REPORT["submission"]["sha256"],
    "oof_auc": RUN_REPORT["ensemble"]["oof_auc"],
    "frozen_oof_auc": RUN_REPORT["ensemble"]["reference_oof_auc"],
    "reproduction_status": RUN_REPORT["reproduction_validation"]["status"],
    "official_competition_score": "NOT_EVALUATED",
}, indent=2))

### Optional post-run verification

Run this only after the full pipeline cell. It verifies run-manifest hashes, every required OOF/test artifact, and submission IDs/schema/probability bounds using the embedded verifier.

In [ ]:
if "RUN_REPORT" not in globals() or "RUN_DIR" not in globals():
    raise RuntimeError("Run the full training cell before post-run verification.")
RUN_VERIFICATION = verify_run(RUN_DIR, DATA_DIR)
submission_frame = __import__("pandas").read_csv(DOWNLOAD_SUBMISSION)
SUBMISSION_VERIFICATION = validate_submission(
    submission_frame, test_raw, sample_submission
)
print(json.dumps({
    "run_verification": RUN_VERIFICATION,
    "submission_verification": SUBMISSION_VERIFICATION,
}, indent=2))